# 🎣 Analyse économique du secteur de la pêche dans l’Union européenne

Le secteur de la pêche européenne est au cœur de tensions majeures : performance économique, dépendance énergétique, durabilité des ressources et cohérence des politiques publiques.

Ce projet vise à transformer des données complexes du STECF (Scientific, Technical and Economic Committee for Fisheries) en un outil d’analyse clair, interactif et exploitable, permettant d’éclairer les décisions publiques et stratégiques.

## 1. Initialisation de l'environnement

Cette étude repose sur l'exploitation de données multidimensionnelles combinant indicateurs économiques et captures biologiques. L'environnement technique est configuré pour supporter le traitement de larges volumes de données et assurer la reproductibilité des analyses :
* **Pandas et Numpy :** manipulation des structures de données complexes

In [85]:
# Importation des librairies pertinentes
import pandas as pd
import numpy as np

# Affichage de toutes les colonnes et lignes (évite les "...")
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)

print("✅ Environnement technique initialisé et configuré.")

✅ Environnement technique initialisé et configuré.


## 2. Chargement des données

Le dataset STECF est distribué sous forme de fichiers Excel multi-onglets contenant des référentiels (codes espèces, géozones) et des séries temporelles fragmentées.
L'approche adoptée ici repose sur une fonction d'importation robuste capable de :
* **Consolider** les données historiques réparties sur plusieurs périodes.
* **Normaliser** les en-têtes et les contenus textuels (nettoyage des espaces blancs).
* **Assurer** la traçabilité des sources via l'injection de la variable origin_sheet.
* **Garantir** l'intégrité en éliminant les enregistrements vides issus du format Excel.

In [86]:
print("⏳ Chargement des données...")

# Configuration de la fonction de chargement des données
def load_excel(path,sheets,cols=None):
    data=pd.read_excel(path,sheet_name=sheets,usecols=cols)
    data=pd.concat([df.assign(origin_sheet=name) for name, df in data.items()],ignore_index=True)
    data.columns=data.columns.astype(str).str.strip().str.lower()
    data=data.apply(lambda x: x.str.strip() if x.dtype=="object" else x)
    cols_to_check=[col for col in data.columns if col !="origin_sheet"]
    data=data.dropna(how="all",subset=cols_to_check).reset_index(drop=True)
    return data

# Configuration des paramètres de chargement

fs_path="Data/Raw/STECF 24-07 - EU Fleet Economic and Transversal data_fleet segment level.xlsx"
fs_sheet=["FS data"]
species_sheet=["codes_species"]
country_sheet=["additional info"]
country_sheet_cols=list(range(0,2))
fishingtech_sheet=["additional info"]
fishingtech_sheet_cols=list(range(4,6))

landings_path="Data/Raw/STECF 24-07 - EU Fleet Economic and Transversal data_landings_FAO.xlsx"
landings_sheet=["Landings 2008-2012","Landings 2013-2017","Landings 2017-"]

species_mapping_path="Data/Raw/ASFIS_sp_2025.xlsx"
species_mapping_sheet=["species_mapping"]

energy_excise_duty_path="Data/Raw/droits_accises_carburant_UE.xlsx"
energy_excise_duty_sheet=["Feuil1"]

# Chargement des données dans les variables respectives
        
fs_raw=load_excel(fs_path,fs_sheet)
species_raw=load_excel(fs_path,species_sheet)
country_raw=load_excel(fs_path,country_sheet,country_sheet_cols)
fishingtech_raw=load_excel(fs_path,fishingtech_sheet,fishingtech_sheet_cols)

landings_raw=load_excel(landings_path,landings_sheet)

species_mapping_raw=load_excel(species_mapping_path,species_mapping_sheet)

energy_excise_duty_raw=load_excel(energy_excise_duty_path,energy_excise_duty_sheet)

print("✅ Fichiers chargés avec succès!")

⏳ Chargement des données...
✅ Fichiers chargés avec succès!


## 3. Prépartion et Nettoyage des données (Data Cleaning)

Cette étape est cruciale pour garantir l'intégrité des analyses sur la flotte de l'UE. 
Les données brutes (data raw) seront transfomés en un dataset exploitable via :
* **L'harmonisation des structures** pour permettre la jointure entre l'économie et les captures (si possible).
* **La validation des types**, pour assurer la précision des calculs de volumes et de valeurs.
* **Le traitement des données manquantes**, particulièrement critiques dans les rapports STECF.

### 3.1 Diagnostic et comparaison des structures

L'harmonisation est l'étape fondatrice qui permet de passer de données brutes disparates à un environnement de travail cohérent. Les rapports du STECF présentent souvent des structures hétérogènes selon qu'ils traitent de la performance économique ou des captures biologiques.

L'enjeu de cette sous-étape est triple :
* **Auditer la structure native :** Identifier les types de données et les métadonnées techniques pour anticiper les erreurs de lecture.
* **Cartographier les divergences :** Repérer les colonnes exclusives à chaque dataset afin de comprendre les différences de granularité (niveaux de détail).
* **Standardiser l'ordre des variables :** Réorganiser les colonnes selon une logique métier commune (allant du général au particulier) pour faciliter la lecture croisée et la maintenance du code.

In [87]:
df_raw={"fs_raw":fs_raw,"landings_raw":landings_raw,"species_raw":species_raw,"species_mapping_raw":species_mapping_raw,"energy_excise_duty_raw":energy_excise_duty_raw,"country_raw":country_raw,"fishingtech_raw":fishingtech_raw}
sorted_names=sorted(df_raw,key=lambda name:df_raw[name].isna().sum().sum()/df_raw[name].size,reverse=True)

# Audit de la structure native
print("------------------- Visualisation technique global des datasets ------------------")

df_list=[]
for name in sorted_names:
    df=df_raw[name]
    nb_cells=df.size
    nb_lines=df.shape[0]
    nb_rows=df.shape[1]
    nb_doublons=df.duplicated().sum()
    nb_na=df.isna().sum().sum()
    pct_na=round((df.isna().sum().sum()/df.size)*100 if nb_cells>0 else 0,2)
    df_list.append({"DF":name,"CELLULES":nb_cells,"LIGNES":nb_lines,"COLONNES":nb_rows,"DOUBLONS":nb_doublons,"NA":nb_na,"% NA":pct_na})
df_audit=pd.DataFrame(df_list)
df_total=pd.DataFrame([{"DF":"TOTAL","CELLULES":df_audit["CELLULES"].sum(),"LIGNES":df_audit["LIGNES"].sum(),"COLONNES":df_audit["COLONNES"].sum(),"DOUBLONS":df_audit["DOUBLONS"].sum(),"NA":df_audit["NA"].sum(),"% NA":df_audit["NA"].sum()/df_audit["CELLULES"].sum()*100}])
audit_df=pd.concat([df_audit,df_total]).reset_index(drop=True)
display((audit_df.style.format({"CELLULES":lambda x: f"{x:,.0f}".replace(",", " "),"LIGNES":lambda x: f"{x:,.0f}".replace(",", " "),"COLONNES":lambda x: f"{x:,.0f}".replace(",", " "),"DOUBLONS":lambda x: f"{x:,.0f}".replace(",", " "),"NA":lambda x: f"{x:,.0f}".replace(",", " "),"% NA":"{:.2f}%"}).hide(axis="index")).background_gradient(subset=["% NA"],cmap="YlOrRd"))

------------------- Visualisation technique global des datasets ------------------


DF,CELLULES,LIGNES,COLONNES,DOUBLONS,NA,% NA
species_mapping_raw,279 260,13 963,20,0,61 046,21.86%
fs_raw,8 488 898,385 859,22,93,1 501 955,17.69%
landings_raw,42 372 750,1 694 910,25,0,6 020 668,14.21%
species_raw,54 808,13 702,4,0,54,0.10%
energy_excise_duty_raw,1 668,278,6,0,0,0.00%
country_raw,75,25,3,0,0,0.00%
fishingtech_raw,45,15,3,0,0,0.00%
TOTAL,51 197 504,2 108 752,83,93,7 583 723,14.81%


L'analyse porte sur **7 tables** totalisant **51 197 504 cellules**. Le diagnostic révèle les indicateurs suivants :
* **Complétude globale :** 85,19% (soit un volume de **NA de 14,81%**)
* **Points de vigilance majeurs :** `species_mapping_raw` (21,86% NA),`fs_raw` (17,69% NA) et `landings_raw` (14,21% NA) présentent les manques les plus critiques, ce qui est significatif compte tenu de leur poids dans l'analyse finale.
* **Doublons détectés :** des redondances ont été identifié, notamment dans fs_raw (93 lignes)

In [88]:
# Paramètrage de la fonction pour vérfier la qualité du dataset
def check_data_quality(df,name="Dataset"):
    print(f"--------------------------------------------------------------- Visualisation technique de {name} ---------------------------------------------------------------")
    audit_cols=pd.DataFrame({"VARIABLE":df.columns,
                             "TYPE":[str(type) for type in df.dtypes],
                             "DOUBLON":[df[col].duplicated().sum() for col in df.columns],
                             "NA":df.isna().sum().to_numpy(),
                             "% NA":(df.isna().sum().to_numpy()/len(df)*100),
                             "MODALITE":df.nunique().to_numpy(),
                             "APERCU":[df[col].unique()[:3].tolist() for col in df.columns]}).sort_values(by="% NA",ascending=False)
    summary_data=pd.DataFrame({"VARIABLE":["--- GLOBAL ---"],
                               "TYPE":"-",
                               "DOUBLON":df.duplicated().sum(),
                               "NA":[df.isna().sum().sum()],
                               "% NA":[(df.isna().sum().sum()/df.size*100)],
                               "MODALITE":"-",
                               "APERCU":[f"{len(df):,.0f}".replace(",", " ")+" lignes et "+f"{df.shape[1]:,.0f}".replace(",", " ")+" colonnes"]
                               })
    audit_final=pd.concat([audit_cols,summary_data],ignore_index=True)
    audit_style=audit_final.style.format({"DOUBLON":lambda x: f"{x:,.0f}".replace(",", " "),"NA":lambda x: f"{x:,.0f}".replace(",", " "),"% NA":"{:.2f}%","MODALITE":lambda x: f"{x:,.0f}".replace(",", " ") if isinstance(x,(int,float)) else x}).background_gradient(subset=["% NA"],cmap="YlOrRd")
    return display(audit_style)

# Visualisation des infos techniques de chaque dataset
for name in sorted_names:
    df_current=df_raw[name]
    check_data_quality(df_current,name)

--------------------------------------------------------------- Visualisation technique de species_mapping_raw ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,russian_name,object,13 370,13 368,95.74%,592,"[nan, 'Буффало', 'Вьюн восточный']"
1,arabic_name,object,11 888,11 889,85.15%,2 074,"[nan, 'شبّوط الوحل', 'لابِييُو']"
2,chinese_name,object,11 363,11 364,81.39%,2 599,"[nan, '大口胭脂鱼', '胭脂鱼']"
3,spanish_name,object,9 126,9 127,65.37%,4 836,"[nan, 'Chupadores NEP', 'Raboseta']"
4,french_name,object,8 215,8 216,58.84%,5 747,"[nan, 'Poisson-taureau', 'Poissons-taureaux NCA']"
5,english_name,object,3 799,3 800,27.21%,10 163,"['Siamese algae-eater', 'River carpsucker', 'Quillback']"
6,author,object,8 189,1 178,8.44%,5 773,"['(Tirant 1883)', '(Rafinesque 1820)', '(Lesueur 1817)']"
7,isscaap_group,float64,13 912,464,3.32%,50,"[11.0, 12.0, 13.0]"
8,family,object,12 670,365,2.61%,1 292,"['GYRINOCHEILIDAE', 'CATOSTOMIDAE', 'BOTIIDAE']"
9,taxonomic_code,object,254,255,1.83%,13 708,"['140001101001', '140002101001', '140002101002']"


--------------------------------------------------------------- Visualisation technique de fs_raw ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,fishery,float64,385 858,385 859,100.00%,0,[nan]
1,activity,object,385 856,374 087,96.95%,2,"[nan, 'A', 'L']"
2,gear,object,385 855,359 327,93.12%,3,"[nan, 'NO', 'LLD']"
3,cluster_name,object,385 537,184 790,47.89%,321,"[nan, 'AREA27 DTS VL2440', 'AREA27 INACTIVE VL2440']"
4,value,float64,256 305,155 350,40.26%,129 553,"[54698472.45, nan, 4529620.35]"
5,geo_indicator,object,385 846,42 542,11.03%,12,"['NGI', nan, 'IC']"
6,upload_date,datetime64[ns],385 776,0,0.00%,83,"[Timestamp('2018-10-01 00:00:00'), Timestamp('2019-04-09 00:00:00'), Timestamp('2019-03-01 00:00:00')]"
7,variable_code,object,385 823,0,0.00%,36,"['totlandginc', 'totrightsinc', 'totdirsub']"
8,template_name,object,385 857,0,0.00%,2,"['map_fs', 'map_capacity']"
9,framework,object,385 858,0,0.00%,1,['map']


--------------------------------------------------------------- Visualisation technique de landings_raw ---------------------------------------------------------------


...

,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,pays,object,250,0,0.00%,28,"['Allemagne', 'Autriche', 'Belgique']"
1,country,object,250,0,0.00%,28,"['Germany', 'Austria', 'Belgium']"
2,country_code,object,250,0,0.00%,28,"['DEU', 'AUT', 'BEL']"
3,year,int64,268,0,0.00%,10,"[2013, 2014, 2015]"
4,tax_rate_eur_per_l,float64,128,0,0.00%,150,"[0.4704, 0.4277, 0.3354]"
5,origin_sheet,object,277,0,0.00%,1,['Feuil1']
6,--- GLOBAL ---,-,0,0,0.00%,-,278 lignes et 6 colonnes


--------------------------------------------------------------- Visualisation technique de country_raw ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,country_code,object,0,0,0.00%,25,"['BEL', 'BGR', 'CCR']"
1,country_name,object,1,0,0.00%,24,"['Belgium', 'Bulgaria', 'test user']"
2,origin_sheet,object,24,0,0.00%,1,['additional info']
3,--- GLOBAL ---,-,0,0,0.00%,-,25 lignes et 3 colonnes


--------------------------------------------------------------- Visualisation technique de fishingtech_raw ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,code,object,0,0,0.00%,15,"['DFN', 'DRB', 'DTS']"
1,description,object,0,0,0.00%,15,"['Drift and/or fixed netters', 'Dredgers', 'Demersal trawlers and/or demersal seiners']"
2,origin_sheet,object,14,0,0.00%,1,['additional info']
3,--- GLOBAL ---,-,0,0,0.00%,-,15 lignes et 3 colonnes


fs_raw et landings_raw sont comparés pour identifier les pivots de fusion et isoler les variables exclusives.
Cette étape prévient la création de colonnes redondantes ou de vides artificiels lors de la jointure des deux tables.

In [89]:
all_cols=sorted(list(set(fs_raw.columns)|set(landings_raw.columns)))
comparison_cols=pd.DataFrame({"Colonne":all_cols,"Dans FS":[c in fs_raw.columns for c in all_cols],"Dans Landings":[c in landings_raw.columns for c in all_cols]})
differences_cols=comparison_cols[comparison_cols["Dans FS"] != comparison_cols["Dans Landings"]]
display(differences_cols)

,Colonne,Dans FS,Dans Landings
12,species_code,False,True
13,species_name,False,True
14,sub_reg,False,True


L'analyse par dataframe révèle des tables de faits et de dimensions :
* **Tables de faits :** 
  * **fs_raw :** **17,69% de NA** sur le dataframe mais avec une situation critique sur la colonne value (**40,26% de NA**)
  * **landings_raw :** **14,21% de NA** sur le data frame mais avec une excellente fiabilité sur la variable value (**0,25% de NA**)
* **Table de dimension :**
  * **species_raw, country_raw, energy_excise_duty_raw et fishingtech_raw :** Tables très propres. *Une donnée de test ("test user") à filtrer dans les pays.*
  * **species_mapping_raw :** NA concentrées sur les traductions linguistiques
  * **geozone_raw :** NA structurels liés à la hiérarchie FAO. Colonne `gsa` totalement vide (100% NA).

Une fusion directe entre `landings_raw` (captures) et `fs_raw` (économie) a été envisagée mais **écarté** pour préserver l'intégrité des données:
*   **Asymétrie de précision** : `fs_raw` opère à l'échelle macro du segment de flotte tandis que `landings_raw` offre **granularité supplémentaire** (détail par expèce)
*   **Apporche retenue** : Maintien de deux tables de faits distinctes partageant des dimensions communes (Modèle en constellation) pour garantir l'exactitude des indicateurs socio-économiques.

Les types des variables sont **validés**

> **PROCHAINE ÉTAPE :** 
> Les résultats révèlent des problématiques de données manquantes hétérogènes. Un entretien avec **Didier Gascuel (Expert SCETF)** sera réalisé pour valider les stratégies de correction et d'imputation des NA.

### 3.2 Audit qualité et nettoyage du contenu

Face au volume important de valeurs manquantes (NA), un entretien avec Didier Gascuel, chercheur spécialiste des pêches maritimes, a été réalisé afin de mieux comprendre les données du STECF et orienter le projet Power BI. Les données utilisées proviennent de l'AER (Annual Economic Report) et sont structurées par pays, taille de navire, type d'engin et année.

**Problèmes identifiés**
* **Forte présence de NA** liée principalement aux règles d'anonymisation :
  * données masquées lorsque les catégories contiennent moins de 10 navires
  * données regroupées dans les plus grandes flottilles
* Les captures seules ne suffisent pas pour analyser l'état des ressources halieutiques
* Les analyses économiques sont biaisées sans prise en compte des subventions publiques

**Solutions et recommandations**
* **Ne pas supprimer automatiquement les NA**
* Utiliser une clé de répartition pour redistribuer les données regroupées et corriger certaines valeurs manquantes
* Exclure les flottilles inactives et l'année 2023 jugée incomplète afin d’éviter de biaiser les analyses économiques
* Supprimer certaines variables jugées peu pertinentes ou trop incomplètes pour l’analyse
* Validation de la classification en 12 flottilles (TRANSIPECHE)
* Intégrer les subventions liées au carburant dans l'analyse
* Principaux constats : 
  * les petits navires sont les plus performants économiquement
  * les grands navires dépendent principalement des aides publiques

Malgré la présence de valeurs manquantes liées aux contraintes d’anonymisation, Didier Gascuel considère que les données restent fiables et font l’objet de mises à jour régulières.

#### 3.2.1 Gestion des valeurs manquantes du dataframe fs_clean

L'assignation de fs_raw vers fs_clean permet de préserver l'intégrité des données sources tout au long du processus de nettoyage.

In [90]:
fs_clean=fs_raw
display(fs_clean.head())

,upload_date,country_name,country_code,year,supra_reg,fishing_tech,vessel_length,geo_indicator,cluster_name,fs_name,variable_group,variable_name,variable_code,value,unit,fromtable,framework,template_name,gear,fishery,activity,origin_sheet
0,2018-10-01,Belgium,BEL,2008,AREA27,TBB,VL2440,NGI,NaN,BEL NAO TBB2440 NGI,Income,Gross value of landings,totlandginc,54698472.45,euro,map_fs,map,map_fs,NaN,NaN,NaN,FS data
1,2018-10-01,Belgium,BEL,2008,AREA27,DTS,VL1012,NGI,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,Income,Gross value of landings,totlandginc,NaN,euro,map_fs,map,map_fs,NaN,NaN,NaN,FS data
2,2018-10-01,Belgium,BEL,2008,AREA27,DTS,VL1824,NGI,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,Income,Gross value of landings,totlandginc,NaN,euro,map_fs,map,map_fs,NaN,NaN,NaN,FS data
3,2018-10-01,Belgium,BEL,2008,AREA27,DTS,VL2440,NGI,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,Income,Gross value of landings,totlandginc,4529620.35,euro,map_fs,map,map_fs,NaN,NaN,NaN,FS data
4,2018-10-01,Belgium,BEL,2008,AREA27,INACTIVE,VL1824,NGI,AREA27 INACTIVE VL2440,BEL NAO INA2440 NGI *,Income,Gross value of landings,totlandginc,NaN,euro,map_fs,map,map_fs,NaN,NaN,NaN,FS data


La réorganisation des colonnes suit une logique d'**entonnoir analytique** : des métadonnées globales (source, date) vers les indicateurs techniques les plus fins (engins, espèces, valeurs).
Cette structure facilite la lecture et prépare les niveaux d'agrégation nécessaires aux futures analyses statistiques.

In [91]:
fs_cols=["origin_sheet","upload_date","fromtable","template_name","framework","cluster_name","fs_name","year","country_code","country_name","supra_reg","geo_indicator","fishery","activity","fishing_tech","vessel_length","gear","variable_group","variable_code","variable_name","unit","value"]
fs_clean=fs_raw.reindex(columns=fs_cols)
display(fs_clean.head(5))

,origin_sheet,upload_date,fromtable,template_name,framework,cluster_name,fs_name,year,country_code,country_name,supra_reg,geo_indicator,fishery,activity,fishing_tech,vessel_length,gear,variable_group,variable_code,variable_name,unit,value
0,FS data,2018-10-01,map_fs,map_fs,map,NaN,BEL NAO TBB2440 NGI,2008,BEL,Belgium,AREA27,NGI,NaN,NaN,TBB,VL2440,NaN,Income,totlandginc,Gross value of landings,euro,54698472.45
1,FS data,2018-10-01,map_fs,map_fs,map,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,2008,BEL,Belgium,AREA27,NGI,NaN,NaN,DTS,VL1012,NaN,Income,totlandginc,Gross value of landings,euro,NaN
2,FS data,2018-10-01,map_fs,map_fs,map,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,2008,BEL,Belgium,AREA27,NGI,NaN,NaN,DTS,VL1824,NaN,Income,totlandginc,Gross value of landings,euro,NaN
3,FS data,2018-10-01,map_fs,map_fs,map,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,2008,BEL,Belgium,AREA27,NGI,NaN,NaN,DTS,VL2440,NaN,Income,totlandginc,Gross value of landings,euro,4529620.35
4,FS data,2018-10-01,map_fs,map_fs,map,AREA27 INACTIVE VL2440,BEL NAO INA2440 NGI *,2008,BEL,Belgium,AREA27,NGI,NaN,NaN,INACTIVE,VL1824,NaN,Income,totlandginc,Gross value of landings,euro,NaN


Le bilan de complétude est établi après réorganisation. Cette vue détaillée identifie les variables critiques dont le taux de valeurs manquantes pourrait impacter la précision des agrégations par pays ou par engins de pêche. Cette étape constitue le **point de départ de la phase de nettoyage** : elle permet d'isoler les variables critiques et d'identifier les zones de forte vacuité pour cibler les traitements prioritaires.

In [92]:
# Visualisation des infos techniques
check_data_quality(fs_clean,"fs_clean")

--------------------------------------------------------------- Visualisation technique de fs_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,fishery,float64,385 858,385 859,100.00%,0,[nan]
1,activity,object,385 856,374 087,96.95%,2,"[nan, 'A', 'L']"
2,gear,object,385 855,359 327,93.12%,3,"[nan, 'NO', 'LLD']"
3,cluster_name,object,385 537,184 790,47.89%,321,"[nan, 'AREA27 DTS VL2440', 'AREA27 INACTIVE VL2440']"
4,value,float64,256 305,155 350,40.26%,129 553,"[54698472.45, nan, 4529620.35]"
5,geo_indicator,object,385 846,42 542,11.03%,12,"['NGI', nan, 'IC']"
6,fs_name,object,384 977,0,0.00%,882,"['BEL NAO TBB2440 NGI', 'BEL NAO DTS2440 NGI *', 'BEL NAO INA2440 NGI *']"
7,year,int64,385 843,0,0.00%,16,"[2008, 2009, 2010]"
8,country_code,object,385 837,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
9,country_name,object,385 837,0,0.00%,22,"['Belgium', 'Bulgaria', 'Cyprus']"


Le dataset est optimisé par l'exclusion des dimensions non pertinentes et le filtrage des observations hors périmètre (vu avec Didier Gascuel) :
* **Formatage temporelle :** Ajustement du format de la variable `year`pour optimiser sa lecture et son traitement dans Power BI.
* **Réduction dimensionnelle :** Suppression des métadonnées techniques et opérationnelles (`upload_date`, `template_name`, etc.) pour se concentrer sur les variables métiers.
* **Filtrage temporel :** Exclusion de l'année 2023 (données potentiellement incomplètes ou provisoires).
* **Ciblage opérationnel :** Suppression des flottes déclarées comme "Inactives" pour ne conserver que l'effort de pêche réel.
* **Bilan post-nettoyage :** Mise à jour du taux de vacuité sur le périmètre final.

In [93]:
# Suppression des variables non pertinentes pour l'analyse
cols_unavaible=["upload_date","fromtable","template_name","framework","cluster_name","geo_indicator","fishery","activity","gear"]
fs_clean=fs_clean.drop(columns=cols_unavaible,errors="ignore")
fs_clean=fs_clean.sort_values(by=["origin_sheet","fs_name","year","country_code","country_name","variable_group","variable_code","variable_name","supra_reg","fishing_tech","vessel_length"]).reset_index(drop=True)

# Exclusion de l'année 2023
fs_clean=fs_clean.loc[fs_clean["year"]!=2023]

# Suppresion des flottes inactives
fs_clean=fs_clean.loc[fs_clean["fishing_tech"].str.upper()!="INACTIVE"]

# Visualisation des infos techniques
check_data_quality(fs_clean,"fs_clean")

--------------------------------------------------------------- Visualisation technique de fs_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,value,float64,173 710,84 428,28.53%,122 223,"[1.0, 12.0, 25.0]"
1,origin_sheet,object,295 933,0,0.00%,1,['FS data']
2,fs_name,object,295 270,0,0.00%,664,"['BEL NAO DTS2440 NGI *', 'BEL NAO PMP1824 NGI *', 'BEL NAO TBB1824 NGI *']"
3,year,int64,295 919,0,0.00%,15,"[2008, 2009, 2010]"
4,country_code,object,295 912,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
5,country_name,object,295 912,0,0.00%,22,"['Belgium', 'Bulgaria', 'Cyprus']"
6,supra_reg,object,295 929,0,0.00%,5,"['AREA27', 'NAO', 'MBS']"
7,fishing_tech,object,295 920,0,0.00%,14,"['DTS', 'PMP', 'DRB']"
8,vessel_length,object,295 924,0,0.00%,10,"['VL1012', 'VL1824', 'VL2440']"
9,variable_group,object,295 928,0,0.00%,6,"['Capacity', 'Capital', 'Effort']"


L'absence de données dans la variable value s'explique par le **secret statistique** (flottilles de moins de 10 navires), où les valeurs sont masquées pour préserver l'anonymat des opérateurs économiques.

Afin de restaurer ces données tout en respectant l'intégrité économique globale (les totaux par pays/année étant connus), un **arbitrage de la clé de répartition** est effectué en comparant la complétude entre l'effort de pêche (kW⋅jours) et le nombre de navires.

In [94]:
fs_na_per_var=fs_clean[fs_clean["value"].isna()].groupby("variable_name").size().sort_values().reset_index().head()
display(fs_na_per_var)
print("Le nombre de navires est retenu comme clé de répartition prioritaire, cette variable étant la plus complète avec un taux de données manquantes quasi nul.")

,variable_name,0
0,Mean LOA of vessels,2
1,Total vessel tonnage,3
2,Total vessel power,3
3,Number of vessels,4
4,Number of fishing trips,1433


Le nombre de navires est retenu comme clé de répartition prioritaire, cette variable étant la plus complète avec un taux de données manquantes quasi nul.


Une table de référence est gérée en se basant sur le poids relatif du nombre de navires pour chaque segment de flotte.
* **Proratisation :** Calcul du ratio Value segment/Total (pays/année)
* **Fiabilisation :** Exclusion des cas où le total est nul pour éviter les divisions par zéro (générant des NA).
* **Contrôle d'intégrité :** Validation de la somme des proportions (doit être égale à 1 par groupe) et vérification de l'absence de doublons ou de valeurs manquantes.

In [95]:
fs_keys_proportion=fs_clean.loc[fs_clean["variable_name"]=="Number of vessels"]
fs_keys_proportion=fs_keys_proportion.groupby(["fs_name","year","fishing_tech","vessel_length","variable_name"])["value"].sum().reset_index()
fs_keys_proportion["total_vessel_fs_year"]=fs_keys_proportion.groupby(["year","fs_name"])["value"].transform("sum")
fs_keys_proportion["keys_proportion"]=(fs_keys_proportion["value"]/fs_keys_proportion["total_vessel_fs_year"]).astype(float)
fs_keys_proportion=fs_keys_proportion.loc[fs_keys_proportion["total_vessel_fs_year"]>0]                                                     # 5 NA ressortaient sur keys_proportion qui est dû au 0 pour total_vessel
display(fs_keys_proportion.sort_values(by=["fs_name","year","vessel_length"]).head(5))

print("Validation de la somme des proportions (Reste=0) :",fs_keys_proportion["keys_proportion"].sum()%1==0)
print("Nombre de doublon détectés :",fs_keys_proportion.duplicated().sum())
print("Nombre de valeurs manquantes :",fs_keys_proportion.isna().sum().sum())

,fs_name,year,fishing_tech,vessel_length,variable_name,value,total_vessel_fs_year,keys_proportion
0,BEL NAO DTS2440 NGI *,2008,DTS,VL1012,Number of vessels,1.0,6.0,0.166667
1,BEL NAO DTS2440 NGI *,2008,DTS,VL1824,Number of vessels,1.0,6.0,0.166667
2,BEL NAO DTS2440 NGI *,2008,DTS,VL2440,Number of vessels,4.0,6.0,0.666667
3,BEL NAO DTS2440 NGI *,2009,DTS,VL1824,Number of vessels,3.0,9.0,0.333333
4,BEL NAO DTS2440 NGI *,2009,DTS,VL2440,Number of vessels,6.0,9.0,0.666667


Validation de la somme des proportions (Reste=0) : True
Nombre de doublon détectés : 0
Nombre de valeurs manquantes : 0


Les données masquées par le secret statistique sont redistribuées au prorata du nombre de navires.
* **Segmentation :** Identification des groupes (Pays/Année/Variable) présentant au moins une valeur manquante via un identifiant de groupe dynamique (ngroup).
* **Ventilation :** Injection des clés de répartition et calcul des new_value. La valeur totale du groupe est redistribuée sur chaque segment de flotte selon son poids relatif.
* **Contrôle de conformité :** Comparaison des masses globales avant et après traitement.
  * Segments stables : Écart nul, confirmant l'absence d'altération des données déjà complètes.
  * Segments ventilés : L'écart constaté valide la "récupération" de la donnée qui était auparavant invisible (somme des NA convertie en valeurs réelles).

In [96]:
# Identification des segments nécessitant une ventilation
fs_clean["id_total_by_fsname"]="total_fs_"+fs_clean.groupby(["fs_name","year","supra_reg","variable_name"]).ngroup().astype(str)
fs_clean["has_na_in_fsname"]=fs_clean.groupby("id_total_by_fsname")["value"].transform(lambda x: x.isna().any() and len(x)>1)
fs_clean["total_by_fsname"]=fs_clean.groupby("id_total_by_fsname")["value"].transform("sum")

# Injection des clés de répartition (Poids statistiques par segment)
fs_clean=fs_clean.drop(columns=["keys_proportion"],errors="ignore")
fs_clean=fs_clean.merge(fs_keys_proportion[["fs_name","year","fishing_tech","vessel_length","keys_proportion"]],on=["fs_name","year","fishing_tech","vessel_length"],how="left",validate="many_to_one")

print(fs_clean["keys_proportion"].isna().sum(),"keys_proportion NA dont",len(fs_clean.loc[(fs_clean["has_na_in_fsname"]==True)&(fs_clean["keys_proportion"].isna())]),"pour les groupes à répartir")
print("Etant donné que les NA se situent sur les groupes ne devant pas faire l'objet de répartition, ces dernières seront non traitées")

# Calcul de la nouvelle distribution (Vérification de la disponibilité des clés)
fs_clean["new_value"]=np.where(fs_clean["has_na_in_fsname"]==True,fs_clean["total_by_fsname"]*fs_clean["keys_proportion"],fs_clean["value"]).round(2)

# Audit de fiabilité : Comparaison Masse initiale vs masse finale
summary_split=fs_clean.groupby("has_na_in_fsname")[["new_value","value"]].sum()
summary_split["variance"]=(summary_split["new_value"]-summary_split["value"]).round(4)
display(summary_split)

159 keys_proportion NA dont 0 pour les groupes à répartir
Etant donné que les NA se situent sur les groupes ne devant pas faire l'objet de répartition, ces dernières seront non traitées


,new_value,value,variance
has_na_in_fsname,,,
False,3.800744e+11,3.800744e+11,1.510000e-02
True,9.221514e+10,9.182338e+10,3.917589e+08


Une vérification de la somme des clés de répartition (censée égaler 100 %) révèle des anomalies localisées sur certains groupes (ex: total_fs_101289).
* **Diagnostic :** Identification de doublons au sein des mêmes segments de flotte. Ces doublons proviennent de lignes redondantes où une valeur réelle coexiste avec un NA pour les mêmes dimensions (year, vessel_length, etc.).
* **Action corrective :** Application d'une règle de priorité visant à conserver la valeur numérique renseignée et à supprimer l'occurrence incomplète. Cette action est essentielle pour garantir que la somme des proratas ne dépasse pas 1 (100 %) et n'introduise pas de biais de surestimation.

In [97]:
# Identification des groupes présentant une anomalie (somme != 1 et !=0)
summary_split_true=fs_clean.loc[fs_clean["has_na_in_fsname"]==True].groupby("id_total_by_fsname")["keys_proportion"].sum().reset_index()
display(summary_split_true.loc[(summary_split_true["keys_proportion"].round(2)!=1.00)&(summary_split_true["keys_proportion"].round(2)!=0.00)].sort_values(by="keys_proportion",ascending=False))

display(fs_clean.loc[fs_clean["id_total_by_fsname"]=="total_fs_101289"])

display(fs_clean.loc[fs_clean["id_total_by_fsname"]=="total_fs_52820"])

,id_total_by_fsname,keys_proportion
27812,total_fs_52826,2.000000
27773,total_fs_52336,2.000000
27782,total_fs_52347,2.000000
27781,total_fs_52346,2.000000
27780,total_fs_52343,2.000000
27779,total_fs_52342,2.000000
27778,total_fs_52341,2.000000
27777,total_fs_52340,2.000000
27776,total_fs_52339,2.000000
27775,total_fs_52338,2.000000


,origin_sheet,fs_name,year,country_code,country_name,supra_reg,fishing_tech,vessel_length,variable_group,variable_code,variable_name,unit,value,id_total_by_fsname,has_na_in_fsname,total_by_fsname,keys_proportion,new_value
156194,FS data,GRC MBS DTS0612 NGI *,2018,GRC,Greece,MBS,DTS,VL0006,Expenditure,totdepcost,Consumption of fixed capital,euro,NaN,total_fs_101289,True,588948.2613,0.00339,1996.43
156195,FS data,GRC MBS DTS0612 NGI *,2018,GRC,Greece,MBS,DTS,VL0612,Expenditure,totdepcost,Consumption of fixed capital,euro,588948.2613,total_fs_101289,True,588948.2613,0.99661,586951.83
156196,FS data,GRC MBS DTS0612 NGI *,2018,GRC,Greece,MBS,DTS,VL0612,Expenditure,totdepcost,Consumption of fixed capital,euro,NaN,total_fs_101289,True,588948.2613,0.99661,586951.83


,origin_sheet,fs_name,year,country_code,country_name,supra_reg,fishing_tech,vessel_length,variable_group,variable_code,variable_name,unit,value,id_total_by_fsname,has_na_in_fsname,total_by_fsname,keys_proportion,new_value
72135,FS data,ESP NAO PS 2440 NGI *,2020,ESP,Spain,NAO,PS,VL2440,Employment,hrworked,Total hours worked per year (engaged crew),hour,1732461.16,total_fs_52820,True,1732461.16,1.0,1732461.16
72136,FS data,ESP NAO PS 2440 NGI *,2020,ESP,Spain,NAO,PS,VL2440,Employment,hrworked,Total hours worked per year (engaged crew),hour,NaN,total_fs_52820,True,1732461.16,1.0,1732461.16


La résolution des conflits structurels suit une approche de **"priorité à la donnée réelle"** :
* **Dédoublonnage intelligent :** Tri des observations pour placer les valeurs nulles en fin de liste (na_position="last"), permettant de ne conserver que la ligne la plus complète lors de la suppression des doublons.
* **Arbitrage de complétude :** Remplacement des valeurs manquantes résiduelles par 0 dans la variable new_value, assurant une base de calcul numérique continue.
* **Validation de clôture :** Un dernier audit de masse confirme la stabilité des totaux et la disparition de la vacuité sur les indicateurs clés.

In [98]:
# Stratégie de nettoyage : tri pour placer les NA en fin de liste et conserver la donnée réelle
cols_to_check=[col for col in fs_clean.columns if col !="value"]
fs_clean=fs_clean.sort_values(by=["id_total_by_fsname","value"],na_position="last")

# Visualisation et suppression des doublons techniques
print("Doublons avec NA à supprimer :",fs_clean.duplicated(subset=cols_to_check).sum())
fs_clean=fs_clean.drop_duplicates(subset=cols_to_check,keep="first").reset_index(drop=True)

# Arbritage final et complétude du dataset
print("NA restants dans la variable new_value après la suppression des doublons :",fs_clean["new_value"].isna().sum())
fs_clean["new_value"]=fs_clean["new_value"].fillna(0)

# Audit de fiabilité : Comparaison Masse initiale vs masse finale
summary_split_final=fs_clean.groupby("has_na_in_fsname")[["new_value","value"]].sum()
summary_split_final["variance"]=(summary_split_final["new_value"]-summary_split_final["value"]).round(4)
display(summary_split_final)

# Visualisation des infos techniques
check_data_quality(fs_clean,"fs_clean")

Doublons avec NA à supprimer : 100
NA restants dans la variable new_value après la suppression des doublons : 8940


,new_value,value,variance
has_na_in_fsname,,,
False,3.800744e+11,3.800744e+11,0.0151
True,9.182338e+10,9.182338e+10,-0.5762


--------------------------------------------------------------- Visualisation technique de fs_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,value,float64,173 610,84 328,28.51%,122 223,"[660950.0, nan, 1224.0]"
1,keys_proportion,float64,294 431,159,0.05%,1 402,"[0.6666666666666666, 0.16666666666666666, 0.1]"
2,origin_sheet,object,295 833,0,0.00%,1,['FS data']
3,fs_name,object,295 170,0,0.00%,664,"['BEL NAO DTS2440 NGI *', 'BEL NAO TBB1824 NGI *', 'BGR MBS TM 1824 NGI']"
4,total_by_fsname,float64,180 582,0,0.00%,115 252,"[660950.0, 1224.0, 241014.0]"
5,has_na_in_fsname,bool,295 832,0,0.00%,2,"[True, False]"
6,id_total_by_fsname,object,102 533,0,0.00%,193 301,"['total_fs_0', 'total_fs_1', 'total_fs_10']"
7,unit,object,295 823,0,0.00%,11,"['euro', 'day', 'metre']"
8,variable_name,object,295 798,0,0.00%,36,"['Consumption of fixed capital', 'Days at sea', 'Investments']"
9,variable_code,object,295 798,0,0.00%,36,"['totdepcost', 'totseadays', 'totinvest']"


La table fs_clean sera finalisé par la substitution des valeurs traitées et l'épuration de l'espace de travail.
* **Substitution :** Remplacement définitif de la colonne value par les données recalculées et ventilées.
* **Optimisation :** Suppression des colonnes temporaires (id_total, keys_proportion, etc.) pour ne conserver que le schéma de données cible, optimisant ainsi la mémoire et la clarté du dataframe.
* **Validation finale :** Le diagnostic de vacuité confirme un dataset opérationnel, sans données manquantes sur les variables quantitatives, prêt pour les phases de visualisation.

In [99]:
# Remplacement de la variable value par la variable new_value
fs_clean["value"]=fs_clean["new_value"]

# Suppression des varibales temporaires
fs_clean=fs_clean.drop(columns=["id_total_by_fsname","has_na_in_fsname","total_by_fsname","keys_proportion","new_value"],errors="ignore")

## Visualisation des infos techniques
check_data_quality(fs_clean,"fs_clean")

print(f"✅ fs_clean est validé. Taille finale : {len(fs_clean):,}".replace(","," ")+" lignes.")

--------------------------------------------------------------- Visualisation technique de fs_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,origin_sheet,object,295 833,0,0.00%,1,['FS data']
1,fs_name,object,295 170,0,0.00%,664,"['BEL NAO DTS2440 NGI *', 'BEL NAO TBB1824 NGI *', 'BGR MBS TM 1824 NGI']"
2,year,int64,295 819,0,0.00%,15,"[2008, 2011, 2010]"
3,country_code,object,295 812,0,0.00%,22,"['BEL', 'BGR', 'GRC']"
4,country_name,object,295 812,0,0.00%,22,"['Belgium', 'Bulgaria', 'Greece']"
5,supra_reg,object,295 829,0,0.00%,5,"['AREA27', 'MBS', 'AREA37']"
6,fishing_tech,object,295 820,0,0.00%,14,"['DTS', 'TBB', 'TM']"
7,vessel_length,object,295 824,0,0.00%,10,"['VL2440', 'VL1012', 'VL1824']"
8,variable_group,object,295 828,0,0.00%,6,"['Expenditure', 'Effort', 'Capital']"
9,variable_code,object,295 798,0,0.00%,36,"['totdepcost', 'totseadays', 'totinvest']"


✅ fs_clean est validé. Taille finale : 295 834 lignes.


Le jeu de données est désormais **stabilisé et exempt de valeurs manquantes**. Cette base saine permet de garantir la fiabilité des analyses statistiques et des visualisations à venir.

#### 3.2.2 Gestion des valeurs manquantes du dataframe landings_clean

L'assignation de landings_raw vers landings_clean permet de préserver l'intégrité des données sources tout au long du processus de nettoyage.

In [100]:
landings_clean=landings_raw
display(landings_clean.head())

,upload_date,country_name,country_code,year,supra_reg,fishing_tech,vessel_length,geo_indicator,gear,fishery,activity,cluster_name,fs_name,variable_group,variable_name,variable_code,value,unit,species_name,species_code,sub_reg,fromtable,framework,template_name,origin_sheet
0,2018-10-01,Belgium,BEL,2008,AREA27,TBB,VL1824,NGI,NaN,NaN,NaN,AREA27 TBB VL1824,BEL NAO TBB1824 NGI *,Landings,Value of landings,totvallandg,139.59,euro,Thornback ray,RJC,27.4.b,map_fsfao,map,map_fsfao,Landings 2008-2012
1,2018-10-01,Belgium,BEL,2008,AREA27,TBB,VL2440,NGI,NaN,NaN,NaN,NaN,BEL NAO TBB2440 NGI,Landings,Value of landings,totvallandg,2698836.79,euro,Common sole,SOL,27.7.f,map_fsfao,map,map_fsfao,Landings 2008-2012
2,2018-10-01,Belgium,BEL,2008,AREA27,DTS,VL2440,NGI,NaN,NaN,NaN,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,Landings,Live weight of landings,totwghtlandg,44652.69,kg,Common sole,SOL,27.7.d,map_fsfao,map,map_fsfao,Landings 2008-2012
3,2018-10-01,Belgium,BEL,2008,AREA27,PMP,VL1824,NGI,NaN,NaN,NaN,AREA27 PMP VL1824,BEL NAO PMP1824 NGI *,Landings,Value of landings,totvallandg,298.61,euro,Lemon sole,LEM,27.4.c,map_fsfao,map,map_fsfao,Landings 2008-2012
4,2018-10-01,Belgium,BEL,2008,AREA27,TBB,VL2440,NGI,NaN,NaN,NaN,NaN,BEL NAO TBB2440 NGI,Landings,Value of landings,totvallandg,11144.66,euro,Atlantic cod,COD,27.7.e,map_fsfao,map,map_fsfao,Landings 2008-2012


La réorganisation des colonnes suit une logique d'**entonnoir analytique** : des métadonnées globales (source, date) vers les indicateurs techniques les plus fins (engins, espèces, valeurs).
Cette structure facilite la lecture et prépare les niveaux d'agrégation nécessaires aux futures analyses statistiques.

In [101]:
landings_cols=["origin_sheet","upload_date","fromtable","template_name","framework","cluster_name","fs_name","year","country_code","country_name","supra_reg","sub_reg","geo_indicator","fishery","activity","fishing_tech","vessel_length","gear","variable_group","variable_code","variable_name","species_code","species_name","unit","value"]
landings_clean=landings_clean.reindex(columns=landings_cols)
display(landings_clean.head())

,origin_sheet,upload_date,fromtable,template_name,framework,cluster_name,fs_name,year,country_code,country_name,supra_reg,sub_reg,geo_indicator,fishery,activity,fishing_tech,vessel_length,gear,variable_group,variable_code,variable_name,species_code,species_name,unit,value
0,Landings 2008-2012,2018-10-01,map_fsfao,map_fsfao,map,AREA27 TBB VL1824,BEL NAO TBB1824 NGI *,2008,BEL,Belgium,AREA27,27.4.b,NGI,NaN,NaN,TBB,VL1824,NaN,Landings,totvallandg,Value of landings,RJC,Thornback ray,euro,139.59
1,Landings 2008-2012,2018-10-01,map_fsfao,map_fsfao,map,NaN,BEL NAO TBB2440 NGI,2008,BEL,Belgium,AREA27,27.7.f,NGI,NaN,NaN,TBB,VL2440,NaN,Landings,totvallandg,Value of landings,SOL,Common sole,euro,2698836.79
2,Landings 2008-2012,2018-10-01,map_fsfao,map_fsfao,map,AREA27 DTS VL2440,BEL NAO DTS2440 NGI *,2008,BEL,Belgium,AREA27,27.7.d,NGI,NaN,NaN,DTS,VL2440,NaN,Landings,totwghtlandg,Live weight of landings,SOL,Common sole,kg,44652.69
3,Landings 2008-2012,2018-10-01,map_fsfao,map_fsfao,map,AREA27 PMP VL1824,BEL NAO PMP1824 NGI *,2008,BEL,Belgium,AREA27,27.4.c,NGI,NaN,NaN,PMP,VL1824,NaN,Landings,totvallandg,Value of landings,LEM,Lemon sole,euro,298.61
4,Landings 2008-2012,2018-10-01,map_fsfao,map_fsfao,map,NaN,BEL NAO TBB2440 NGI,2008,BEL,Belgium,AREA27,27.7.e,NGI,NaN,NaN,TBB,VL2440,NaN,Landings,totvallandg,Value of landings,COD,Atlantic cod,euro,11144.66


Le dataset est optimisé par l'exclusion des dimensions non pertinentes et le filtrage des observations hors périmètre (vu avec Didier Gascuel) :
* **Réduction dimensionnelle :** Suppression des métadonnées techniques et opérationnelles (upload_date, template_name, etc.) pour se concentrer sur les variables métiers.
* **Filtrage temporel :** Exclusion de l'année 2023 (données potentiellement incomplètes ou provisoires).
* **Ciblage opérationnel :** Suppression des flottes déclarées comme "Inactives" pour ne conserver que l'effort de pêche réel.
* **Anlayse croisée des codes espèces :** Qualification des espèces manquantes en "Unknown". L'absence des noms dans les deux sources (données et référentiel), rend toute récupération impossible via les codes espèces.
* **Bilan post-nettoyage :** Mise à jour du taux de vacuité sur le périmètre final.

In [102]:
# Suppresion des variables non pertinentes
cols_unavaible=["upload_date","fromtable","template_name","framework","cluster_name","geo_indicator","fishery","activity","gear"]
landings_clean=landings_clean.drop(columns=cols_unavaible,errors="ignore")
display(landings_clean.head())
landings_clean=landings_clean.sort_values(by=["origin_sheet","fs_name","year","country_code","country_name","variable_group","variable_code","variable_name","supra_reg","sub_reg","fishing_tech","vessel_length","species_code","species_name"]).reset_index(drop=True)

# Exclusion de l'année 2023
landings_clean=landings_clean.loc[landings_clean["year"]!=2023]

# Suppresion des flottes inactives
landings_clean=landings_clean.loc[landings_clean["fishing_tech"].str.upper()!="INACTIVE"]

# Remplacement des NA de la variable species_name
landings_na_species=set(landings_clean.loc[landings_clean["species_name"].isna(),"species_code"].unique())
species_dispo=species_raw.loc[species_raw["common_name"].notna(),"code"].unique()
na_species_reparables=landings_na_species.intersection(species_dispo)
print(f"Nombre de codes récupérables: {len(na_species_reparables)}. Ce résultat signifie que les données manquantes sont structurelles : le code existe bien dans les deux tables mais le nom est manquant. Au vu du faible taux de NA, ces dernières seront remplacées par Unknown")
landings_clean["species_name"]=landings_clean["species_name"].fillna("Unknown")

# Visualisation des infos techniques
check_data_quality(landings_clean,"landings_clean")


,origin_sheet,fs_name,year,country_code,country_name,supra_reg,sub_reg,fishing_tech,vessel_length,variable_group,variable_code,variable_name,species_code,species_name,unit,value
0,Landings 2008-2012,BEL NAO TBB1824 NGI *,2008,BEL,Belgium,AREA27,27.4.b,TBB,VL1824,Landings,totvallandg,Value of landings,RJC,Thornback ray,euro,139.59
1,Landings 2008-2012,BEL NAO TBB2440 NGI,2008,BEL,Belgium,AREA27,27.7.f,TBB,VL2440,Landings,totvallandg,Value of landings,SOL,Common sole,euro,2698836.79
2,Landings 2008-2012,BEL NAO DTS2440 NGI *,2008,BEL,Belgium,AREA27,27.7.d,DTS,VL2440,Landings,totwghtlandg,Live weight of landings,SOL,Common sole,kg,44652.69
3,Landings 2008-2012,BEL NAO PMP1824 NGI *,2008,BEL,Belgium,AREA27,27.4.c,PMP,VL1824,Landings,totvallandg,Value of landings,LEM,Lemon sole,euro,298.61
4,Landings 2008-2012,BEL NAO TBB2440 NGI,2008,BEL,Belgium,AREA27,27.7.e,TBB,VL2440,Landings,totvallandg,Value of landings,COD,Atlantic cod,euro,11144.66


Nombre de codes récupérables: 0. Ce résultat signifie que les données manquantes sont structurelles : le code existe bien dans les deux tables mais le nom est manquant. Au vu du faible taux de NA, ces dernières seront remplacées par Unknown
--------------------------------------------------------------- Visualisation technique de landings_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,value,float64,912 413,4 192,0.25%,763 020,"[1253.16, 33.5, 7240.72]"
1,origin_sheet,object,1 675 431,0,0.00%,3,"['Landings 2008-2012', 'Landings 2013-2017', 'Landings 2017-']"
2,fs_name,object,1 674 791,0,0.00%,643,"['BEL NAO DTS2440 NGI *', 'BEL NAO PMP1824 NGI *', 'BEL NAO TBB1824 NGI *']"
3,year,int64,1 675 419,0,0.00%,15,"[2008, 2009, 2010]"
4,country_code,object,1 675 412,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
5,country_name,object,1 675 412,0,0.00%,22,"['Belgium', 'Bulgaria', 'Cyprus']"
6,supra_reg,object,1 675 429,0,0.00%,5,"['AREA27', 'MBS', 'AREA37']"
7,sub_reg,object,1 675 181,0,0.00%,253,"['27.4.b', '27.4.c', '27.7.a']"
8,fishing_tech,object,1 675 420,0,0.00%,14,"['DTS', 'PMP', 'TBB']"
9,vessel_length,object,1 675 424,0,0.00%,10,"['VL2440', 'VL1824', 'VL0006']"


L'analyse de l'impact des valeurs manquantes et l'arbitrage sur la fiabilité de la variable `value` sont les suivants :
* **Diagnostic de concentration :** L'analyse révèle que **60% des NA** sont localisés sur la flotte espagnole pour des années spécifiques (2008, 2009, 2015, 2017).
* **Évaluation de l'impact :** Ces manques ne représentent que 0,25% de la masse de la variable `value` et 0,02% du volume total du dataset.
* **Arbitrage :** Le volume étant statistiquement non significatif, la **suppression des lignes** est privilégiée. Cette décision garantit un dataset final 100% numérique sans introduire de biais majeur.
* **Validation :** Le contrôle final confirme l'absence totale de données manquantes, validant le dataset pour les phases de calculs agrégés.

In [103]:
# Analyse de la provenance des NA par pays et année
landings_clean["is_na"]=landings_clean["value"].isna()
pivot_na=landings_clean.groupby(["country_name","year"])["is_na"].sum()/landings_clean["is_na"].sum()*100
display(pivot_na[pivot_na>0].sort_values(ascending=False).head(10))
print("Analyse de l'impact relatif de l'Espagne :",pivot_na.loc["Spain"].sum().round(2),"%")

# Suppression des NA et suppression de la colonne temporaire
landings_clean=landings_clean.dropna(subset="value").reset_index(drop=True)
landings_clean=landings_clean.drop(columns="is_na")

# Visualisation des infos techniques
check_data_quality(landings_clean,"landings_clean")

print(f"✅ Landings_clean est validé. Taille finale : {len(landings_clean):,}".replace(","," ")+" lignes.")

country_name  year
Spain         2015    31.488550
              2017    18.225191
              2008     5.629771
              2009     5.415076
Ireland       2012     3.602099
              2013     2.910305
Poland        2020     2.814885
Ireland       2011     2.552481
Poland        2016     2.099237
Ireland       2014     2.027672
Name: is_na, dtype: float64

Analyse de l'impact relatif de l'Espagne : 60.76 %
--------------------------------------------------------------- Visualisation technique de landings_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,origin_sheet,object,1 671 239,0,0.00%,3,"['Landings 2008-2012', 'Landings 2013-2017', 'Landings 2017-']"
1,fs_name,object,1 670 599,0,0.00%,643,"['BEL NAO DTS2440 NGI *', 'BEL NAO PMP1824 NGI *', 'BEL NAO TBB1824 NGI *']"
2,year,int64,1 671 227,0,0.00%,15,"[2008, 2009, 2010]"
3,country_code,object,1 671 220,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
4,country_name,object,1 671 220,0,0.00%,22,"['Belgium', 'Bulgaria', 'Cyprus']"
5,supra_reg,object,1 671 237,0,0.00%,5,"['AREA27', 'MBS', 'AREA37']"
6,sub_reg,object,1 670 989,0,0.00%,253,"['27.4.b', '27.4.c', '27.7.a']"
7,fishing_tech,object,1 671 228,0,0.00%,14,"['DTS', 'PMP', 'TBB']"
8,vessel_length,object,1 671 232,0,0.00%,10,"['VL2440', 'VL1824', 'VL0006']"
9,variable_group,object,1 671 241,0,0.00%,1,['Landings']


✅ Landings_clean est validé. Taille finale : 1 671 242 lignes.


Le jeu de données est désormais **stabilisé et exempt de valeurs manquantes**. Cette base saine permet de garantir la fiabilité des analyses statistiques et des visualisations à venir.

#### 3.2.3 Gestion des valeurs manquantes du dataframe species_raw

L'assignation de species_raw vers species_clean permet de préserver l'intégrité des données sources tout au long du processus de nettoyage.

In [104]:
species_clean=species_raw
display(species_clean.head(5))

,code,scientific_name,common_name,origin_sheet
0,AAA,Acipenser naccarii,Adriatic sturgeon,codes_species
1,AAB,Acanthopagrus bifasciatus,Twobar seabream,codes_species
2,AAC,Amia calva,Bowfin,codes_species
3,AAD,Acipenser dabryanus,Yangtze sturgeon,codes_species
4,AAE,Antennarius analis,Tailjet frogfish,codes_species


La réorganisation des colonnes suit une logique d'**entonnoir analytique** (Du plus précis au plus global).
Cette structure facilite la lecture et prépare les niveaux d'agrégation nécessaires aux futures analyses statistiques.

In [105]:
species_clean=species_clean.rename(columns={"code":"species_code","common_name":"species_name","scientific_name":"species_scientificname"})
species_cols=["species_code","species_name","species_scientificname","origin_sheet"]
species_clean=species_clean.reindex(columns=species_cols).sort_values(by="species_code")
display(species_clean.head(5))

,species_code,species_name,species_scientificname,origin_sheet
0,AAA,Adriatic sturgeon,Acipenser naccarii,codes_species
1,AAB,Twobar seabream,Acanthopagrus bifasciatus,codes_species
2,AAC,Bowfin,Amia calva,codes_species
3,AAD,Yangtze sturgeon,Acipenser dabryanus,codes_species
4,AAE,Tailjet frogfish,Antennarius analis,codes_species


Avant l'intégration, nous vérifions la qualité du référentiel afin d'assurer l'exhaustivité des jointures futures.
* **Contrôle d'unicité :** Vérification de l'absence de doublons pour éviter toute explosion du nombre de lignes lors des rapprochements de tables.
* **Analyse de la structure :** Examen des types de données et de la complétude des champs.
* **Diagnostic de vacuité :** Identification des espèces non renseignées qui pourraient entraîner une perte d'information lors de l'analyse.

In [106]:
# Visualisation des infos techniques
check_data_quality(species_clean,"species_clean")

--------------------------------------------------------------- Visualisation technique de species_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,species_name,object,70,54,0.39%,13 631,"['Adriatic sturgeon', 'Twobar seabream', 'Bowfin']"
1,species_code,object,0,0,0.00%,13 702,"['AAA', 'AAB', 'AAC']"
2,species_scientificname,object,14,0,0.00%,13 688,"['Acipenser naccarii', 'Acanthopagrus bifasciatus', 'Amia calva']"
3,origin_sheet,object,13 701,0,0.00%,1,['codes_species']
4,--- GLOBAL ---,-,0,54,0.10%,-,13 702 lignes et 4 colonnes


La tentative de récupération des étiquettes et sécurisation du référentiel sont réalisées :
* **Lookup de secours :** Tentative d'enrichissement de common_name en utilisant les correspondances réelles observées dans le dataset landings_clean.
* **Constat technique :** Les valeurs manquantes sont confirmées comme structurelles (absentes des deux sources), rendant toute récupération automatique par croisement impossible.
* **Arbitrage final :** Remplacement des valeurs résiduelles par "Unknown". Cette action neutralise le risque de perte de données lors des regroupements statistiques tout en signalant explicitement les lacunes du référentiel d'origine.

In [107]:
# Tentative de remplacement des NA par la variabe species_name de landings_clean
lookup_species=landings_clean.loc[landings_clean["species_name"]!="Unknown",["species_code","species_name"]]
lookup_species=lookup_species.drop_duplicates(subset="species_code")
species_clean["species_name"]=species_clean["species_name"].fillna(species_clean["species_code"].map(lookup_species.set_index("species_code")["species_name"]))
print("NA restants dans species_name :",species_clean["species_name"].isna().sum())
print("Les NA sont structurels donc le remplacement par la variable species_name est impossible.")

# Remplacement des NA par Unknwon
species_clean["species_name"]=species_clean["species_name"].fillna("Unknown")


NA restants dans species_name : 54
Les NA sont structurels donc le remplacement par la variable species_name est impossible.


In [108]:
# Visualisation des infos techniques
check_data_quality(species_clean,"species_clean")

print(f"✅ species_clean est validé. Taille finale : {len(species_clean):,}".replace(","," ")+" lignes.")

--------------------------------------------------------------- Visualisation technique de species_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,species_code,object,0,0,0.00%,13 702,"['AAA', 'AAB', 'AAC']"
1,species_name,object,71,0,0.00%,13 631,"['Adriatic sturgeon', 'Twobar seabream', 'Bowfin']"
2,species_scientificname,object,14,0,0.00%,13 688,"['Acipenser naccarii', 'Acanthopagrus bifasciatus', 'Amia calva']"
3,origin_sheet,object,13 701,0,0.00%,1,['codes_species']
4,--- GLOBAL ---,-,0,0,0.00%,-,13 702 lignes et 4 colonnes


✅ species_clean est validé. Taille finale : 13 702 lignes.


Le jeu de données est désormais **stabilisé et exempt de valeurs manquantes**. Cette base saine permet de garantir la fiabilité des analyses statistiques et des visualisations à venir.

### 3.2.4 Gestion des valeurs manquantes du dataframe species_mapping_raw

L'assignation de species_mapping_raw vers species_mapping_clean permet de préserver l'intégrité des données sources tout au long du processus de nettoyage.

In [109]:
species_mapping_clean=species_mapping_raw
display(species_mapping_clean.head(5))

,isscaap_group,taxonomic_code,alpha3_code,scientific_name,english_name,french_name,spanish_name,arabic_name,chinese_name,russian_name,author,family,order or higher taxa,fishstat_data,categorie_fr,category_en,species_category_fr,species_category,species_subcategory,origin_sheet
0,11.0,140001101001,GXM,Gyrinocheilus aymonieri,Siamese algae-eater,NaN,NaN,NaN,NaN,NaN,(Tirant 1883),GYRINOCHEILIDAE,CYPRINIFORMES,NO,Poissons d'eau douce,Freshwater fish,Poissons d'eau douce,Fishs,Freshwater fish,species_mapping
1,11.0,140002101001,CDO,Carpiodes carpio,River carpsucker,NaN,NaN,NaN,NaN,NaN,(Rafinesque 1820),CATOSTOMIDAE,CYPRINIFORMES,YES,Poissons d'eau douce,Freshwater fish,Poissons d'eau douce,Fishs,Freshwater fish,species_mapping
2,11.0,140002101002,FMP,Carpiodes cyprinus,Quillback,NaN,NaN,NaN,NaN,NaN,(Lesueur 1817),CATOSTOMIDAE,CYPRINIFORMES,NO,Poissons d'eau douce,Freshwater fish,Poissons d'eau douce,Fishs,Freshwater fish,species_mapping
3,11.0,140002101801,ATC,Catostomus catostomus,Longnose sucker,NaN,NaN,NaN,NaN,NaN,(Forster 1773),CATOSTOMIDAE,CYPRINIFORMES,NO,Poissons d'eau douce,Freshwater fish,Poissons d'eau douce,Fishs,Freshwater fish,species_mapping
4,11.0,140002101802,ATO,Catostomus commersonii,White sucker,NaN,NaN,NaN,NaN,NaN,(Lacépède 1803),CATOSTOMIDAE,CYPRINIFORMES,YES,Poissons d'eau douce,Freshwater fish,Poissons d'eau douce,Fishs,Freshwater fish,species_mapping


Avant l'intégration, nous vérifions la qualité du référentiel afin d'assurer l'exhaustivité des jointures futures.
* **Contrôle d'unicité :** Vérification de l'absence de doublons pour éviter toute explosion du nombre de lignes lors des rapprochements de tables.
* **Analyse de la structure :** Examen des types de données et de la complétude des champs de segmentation.
* **Diagnostic de vacuité :** Identification des espèces non renseignées qui pourraient entraîner une perte d'information lors de l'analyse.

In [110]:
# Visualisation des infos techniques
check_data_quality(species_mapping_clean,"species_mapping_clean")

--------------------------------------------------------------- Visualisation technique de species_mapping_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,russian_name,object,13 370,13 368,95.74%,592,"[nan, 'Буффало', 'Вьюн восточный']"
1,arabic_name,object,11 888,11 889,85.15%,2 074,"[nan, 'شبّوط الوحل', 'لابِييُو']"
2,chinese_name,object,11 363,11 364,81.39%,2 599,"[nan, '大口胭脂鱼', '胭脂鱼']"
3,spanish_name,object,9 126,9 127,65.37%,4 836,"[nan, 'Chupadores NEP', 'Raboseta']"
4,french_name,object,8 215,8 216,58.84%,5 747,"[nan, 'Poisson-taureau', 'Poissons-taureaux NCA']"
5,english_name,object,3 799,3 800,27.21%,10 163,"['Siamese algae-eater', 'River carpsucker', 'Quillback']"
6,author,object,8 189,1 178,8.44%,5 773,"['(Tirant 1883)', '(Rafinesque 1820)', '(Lesueur 1817)']"
7,isscaap_group,float64,13 912,464,3.32%,50,"[11.0, 12.0, 13.0]"
8,family,object,12 670,365,2.61%,1 292,"['GYRINOCHEILIDAE', 'CATOSTOMIDAE', 'BOTIIDAE']"
9,taxonomic_code,object,254,255,1.83%,13 708,"['140001101001', '140002101001', '140002101002']"


Pour l'enrichissement de la table species_raw, seules les variables suivantes sont conservées : `alpha3_code`,`scientific_name`,`family`,`order or higher taxa` et `species_category`. Les autres variables seront supprimées.
Pour les valeurs manquantes de la variable `family` et `order or higher taxa`, ces dernières seront respectivement remplacées par "Unidentified family" et "Unidentified order or higher taxa"

In [111]:
# Conservation des colonnes pertinents
species_mapping_clean=species_mapping_clean[["alpha3_code","family","order or higher taxa","species_category"]]
species_mapping_clean=species_mapping_clean.sort_values(by="alpha3_code").reset_index(drop=True)

# Remplacement des NA de la variable family et order or higher taxa
species_mapping_clean["family"]=species_mapping_clean["family"].fillna("Unidentified family")
species_mapping_clean["order or higher taxa"]=species_mapping_clean["order or higher taxa"].fillna("Unidentified order or higher taxa")

# Visualisation des infos techniques
check_data_quality(species_mapping_clean,"species_mapping_clean")

print(f"✅ species_mapping_clean est validé. Taille finale : {len(species_mapping_clean):,}".replace(","," ")+" lignes.")

--------------------------------------------------------------- Visualisation technique de species_mapping_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,alpha3_code,object,0,0,0.00%,13 963,"['AAA', 'AAB', 'AAC']"
1,family,object,12 670,0,0.00%,1 293,"['ACIPENSERIDAE', 'SPARIDAE', 'AMIIDAE']"
2,order or higher taxa,object,13 757,0,0.00%,206,"['ACIPENSERIFORMES', 'ACANTHURIFORMES', 'AMIIFORMES']"
3,species_category,object,13 958,0,0.00%,5,"['Fishs', 'Crustaceans', 'Molluscs & Invertebrates']"
4,--- GLOBAL ---,-,0,0,0.00%,-,13 963 lignes et 4 colonnes


✅ species_mapping_clean est validé. Taille finale : 13 963 lignes.


Le jeu de données est désormais **stabilisé et exempt de valeurs manquantes**. Cette base saine permet de garantir la fiabilité des analyses statistiques et des visualisations à venir.

### 3.2.5 Gestion des valeurs manquantes des dataframes country_raw, fishingtech_raw et energy_excise_duty_raw

L'assignation de country_raw, fishingtech_raw et energy_excise_duty_raw vers respectivement country_clean, fishingtech_clean et energy_excise_duty_clean permet de préserver l'intégrité des données sources tout au long du processus de nettoyage.

In [112]:
country_clean=country_raw
fishingtech_clean=fishingtech_raw
energy_excise_duty_clean=energy_excise_duty_raw
display(country_clean.head(5))
display(fishingtech_clean.head(5))
display(energy_excise_duty_clean.head(5))

,country_code,country_name,origin_sheet
0,BEL,Belgium,additional info
1,BGR,Bulgaria,additional info
2,CCR,test user,additional info
3,CYP,Cyprus,additional info
4,DEU,Germany,additional info


,code,description,origin_sheet
0,DFN,Drift and/or fixed netters,additional info
1,DRB,Dredgers,additional info
2,DTS,Demersal trawlers and/or demersal seiners,additional info
3,FPO,Vessels using pots and/or traps,additional info
4,HOK,Vessels using hooks,additional info


,pays,country,country_code,year,tax_rate_eur_per_l,origin_sheet
0,Allemagne,Germany,DEU,2013,0.4704,Feuil1
1,Autriche,Austria,AUT,2013,0.4277,Feuil1
2,Belgique,Belgium,BEL,2013,0.4277,Feuil1
3,Bulgarie,Bulgaria,BGR,2013,0.3354,Feuil1
4,Chypre,Cyprus,CYP,2013,0.4607,Feuil1


L'audit de la structure de référence assure la cohérence des axes d'analyse socio-économiques :
* **Référentiel Pays (country_clean) :** Validation des codes ISO et des noms de pays. Cette étape garantit une cartographie sans erreur des débarquements et des flottes par État membre.
* **Référentiel Technique (fishingtech_clean) :** Fiabilisation des libellés d'engins de pêche.
* **Référentiel fiscal (energy_excise_duty_clean) :** Intégration et normalisation des données relatives aux droits d'accise sur les produits énergétiques.
* **Intégrité globale :** Avec un taux de vacuité de 0% sur l'ensemble des dimensions, le système est prêt pour une consolidation multidimensionnelle sans perte d'information.

In [113]:
# Suppression des variables non pertinentes de energy_excise_duty_clean
energy_excise_duty_clean=energy_excise_duty_clean.drop("pays",axis=1,errors="ignore")

# Renommage des colonnes de energy_excise_duty_clean
energy_excise_duty_clean=energy_excise_duty_clean.rename(columns={"country":"country_name","tax_rate_eur_per_l":"excise_duty_eur_l"})
energy_excise_duty_clean=energy_excise_duty_clean[["year","country_code","country_name","excise_duty_eur_l","origin_sheet"]]

# Visualisation des infos techniques
check_data_quality(energy_excise_duty_clean,"energy_excise_duty_clean")

print(f"✅ energy_excise_duty_clean est validé. Taille finale : {len(energy_excise_duty_clean):,}".replace(","," ")+" lignes.")

# Suppresion des lignes non pertinentes
country_clean=country_clean.loc[(country_clean["country_code"]!="JRC") & (country_clean["country_code"]!="STF") & (country_clean["country_code"]!="CCR")]

# Visualisation des infos techniques
check_data_quality(country_clean,"country_clean")

print(f"✅ country_clean est validé. Taille finale : {len(country_clean):,}".replace(","," ")+" lignes.")

# Suppression des lignes non pertinentes
fishingtech_clean=fishingtech_clean.loc[(fishingtech_clean["code"]!="INACTIVE")]

# Renommage des colonnes de fishingtech_clean
fishingtech_clean=fishingtech_clean.rename(columns={"code":"fishingtech_code","description":"fishingtech_name"})
fishingtech_cols=["fishingtech_code","fishingtech_name","origin_sheet"]
fishingtech_clean=fishingtech_clean.reindex(columns=fishingtech_cols).sort_values(by="fishingtech_code")
display(fishingtech_clean.head(5))

# Visualisation des infos techniques
check_data_quality(fishingtech_clean,"fishingtech_clean")

print(f"✅ fishingtech_clean est validé. Taille finale : {len(fishingtech_clean):,}".replace(","," ")+" lignes.")

--------------------------------------------------------------- Visualisation technique de energy_excise_duty_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,year,int64,268,0,0.00%,10,"[2013, 2014, 2015]"
1,country_code,object,250,0,0.00%,28,"['DEU', 'AUT', 'BEL']"
2,country_name,object,250,0,0.00%,28,"['Germany', 'Austria', 'Belgium']"
3,excise_duty_eur_l,float64,128,0,0.00%,150,"[0.4704, 0.4277, 0.3354]"
4,origin_sheet,object,277,0,0.00%,1,['Feuil1']
5,--- GLOBAL ---,-,0,0,0.00%,-,278 lignes et 5 colonnes


✅ energy_excise_duty_clean est validé. Taille finale : 278 lignes.
--------------------------------------------------------------- Visualisation technique de country_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,country_code,object,0,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
1,country_name,object,0,0,0.00%,22,"['Belgium', 'Bulgaria', 'Cyprus']"
2,origin_sheet,object,21,0,0.00%,1,['additional info']
3,--- GLOBAL ---,-,0,0,0.00%,-,22 lignes et 3 colonnes


✅ country_clean est validé. Taille finale : 22 lignes.


,fishingtech_code,fishingtech_name,origin_sheet
0,DFN,Drift and/or fixed netters,additional info
1,DRB,Dredgers,additional info
2,DTS,Demersal trawlers and/or demersal seiners,additional info
3,FPO,Vessels using pots and/or traps,additional info
4,HOK,Vessels using hooks,additional info


--------------------------------------------------------------- Visualisation technique de fishingtech_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,fishingtech_code,object,0,0,0.00%,14,"['DFN', 'DRB', 'DTS']"
1,fishingtech_name,object,0,0,0.00%,14,"['Drift and/or fixed netters', 'Dredgers', 'Demersal trawlers and/or demersal seiners']"
2,origin_sheet,object,13,0,0.00%,1,['additional info']
3,--- GLOBAL ---,-,0,0,0.00%,-,14 lignes et 3 colonnes


✅ fishingtech_clean est validé. Taille finale : 14 lignes.


Le jeu de données est désormais **stabilisé et exempt de valeurs manquantes**. Cette base saine permet de garantir la fiabilité des analyses statistiques et des visualisations à venir.

## 4.Alimentation et Exportation du modèle relationnel

Cette étape consiste à charger les données définitives dnas les tables de dimensions et de faits, en garantissant l'unicité des références pour Power BI.

### 4.1 Alimentation des tables de dimensions

#### 4.1.1 Alimentation de la table de dimension species_clean

Face à la diversité des 13 702 espèces recensées, une catégorisation s’impose pour optimiser la lisibilité des visualisations. En s'appuyant sur les regroupements taxonomiques par famille de la **FAO** (Food and Agriculture Organization), les espèces ont été réparties en **5 catégories clés**, offrant ainsi une structure d'analyse pertinente et cohérente.

In [114]:
# Fusion des deux tables
cols_to_map=["family","order or higher taxa", "species_category"]
species_clean=species_clean.drop(columns=[col for col in cols_to_map if col in species_clean.columns],errors="ignore")
species_clean=species_clean.merge(species_mapping_clean.set_index("alpha3_code")[cols_to_map],left_on="species_code",right_index=True,how="left")[["species_code","species_name","species_scientificname","family","order or higher taxa","species_category","origin_sheet"]]

# Visualisation des infos techniques
check_data_quality(species_clean,"species_clean")

# Visualisation de species_clean
display(species_clean)

--------------------------------------------------------------- Visualisation technique de species_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,species_code,object,0,0,0.00%,13 702,"['AAA', 'AAB', 'AAC']"
1,species_name,object,71,0,0.00%,13 631,"['Adriatic sturgeon', 'Twobar seabream', 'Bowfin']"
2,species_scientificname,object,14,0,0.00%,13 688,"['Acipenser naccarii', 'Acanthopagrus bifasciatus', 'Amia calva']"
3,family,object,12 428,0,0.00%,1 274,"['ACIPENSERIDAE', 'SPARIDAE', 'AMIIDAE']"
4,order or higher taxa,object,13 502,0,0.00%,200,"['ACIPENSERIFORMES', 'ACANTHURIFORMES', 'AMIIFORMES']"
5,species_category,object,13 697,0,0.00%,5,"['Fishs', 'Crustaceans', 'Molluscs & Invertebrates']"
6,origin_sheet,object,13 701,0,0.00%,1,['codes_species']
7,--- GLOBAL ---,-,0,0,0.00%,-,13 702 lignes et 7 colonnes


,species_code,species_name,species_scientificname,family,order or higher taxa,species_category,origin_sheet
0,AAA,Adriatic sturgeon,Acipenser naccarii,ACIPENSERIDAE,ACIPENSERIFORMES,Fishs,codes_species
1,AAB,Twobar seabream,Acanthopagrus bifasciatus,SPARIDAE,ACANTHURIFORMES,Fishs,codes_species
2,AAC,Bowfin,Amia calva,AMIIDAE,AMIIFORMES,Fishs,codes_species
3,AAD,Yangtze sturgeon,Acipenser dabryanus,ACIPENSERIDAE,ACIPENSERIFORMES,Fishs,codes_species
4,AAE,Tailjet frogfish,Antennarius analis,ANTENNARIIDAE,LOPHIIFORMES,Fishs,codes_species
...,...,...,...,...,...,...,...
13697,ZYZ,Zaireichthys zonatus,Zaireichthys zonatus,AMPHILIIDAE,SILURIFORMES,Fishs,codes_species
13698,ZZB,Clarias nigromarmoratus,Clarias nigromarmoratus,CLARIIDAE,SILURIFORMES,Fishs,codes_species
13699,ZZC,Dark-mouth chimaera,Chimaera buccanigella,CHIMAERIDAE,CHIMAERIFORMES,Fishs,codes_species
13700,ZZD,Falkor chimaera,Chimaera didierae,CHIMAERIDAE,CHIMAERIFORMES,Fishs,codes_species


#### 4.1.2 Alimentation de la table de dimension fishingtech_clean

Les techniques de pêche vont être regroupé en 5 catégories selon les référentiels de la pêche :
* **PG - Passive gears (nets, lines, traps) :** Dor - Filets, lignes, casiers
* **Dre - Dredgers and polyvalent :** Dra - Dragues et polyvalents
* **TraD - Demersal trawlers and seiners :** ChaD - Chaluts et sennes démersaux
* **TraP - Pelagic trawlers and seiners :**ChaP - Chaluts et sennes pélagiques

In [115]:
# Nomenclature des techniques de pêche
dict_fishingtech={
    "PG":({"DFN","FPO","HOK","MGO","PGO","PGP","PG"},"Passive gears (nets, lines, traps)"),
    "Dre":({"DRB","MGP","PMP"},"Dredgers and polyvalent"),
    "TraD":({"DTS","TBB"},"Demersal trawlers and seiners"),
    "TraP":({"PS","TM","TMP"},"Pelagic trawlers and seiners"),
    "INACTIVE":({"INACTIVE"},"INACTIVE")
}

# Dictionnaire de correspondance
fishingtech_code_map={code: key for key,(codes,_) in dict_fishingtech.items() for code in codes}
fishingtech_name_map={code: name for _,(codes,name) in dict_fishingtech.items() for code in codes}

# Mapping sur le DataFrame
fishingtech_clean["fishingtech_category_code"]=fishingtech_clean["fishingtech_code"].map(fishingtech_code_map).fillna("INACTIVE")
fishingtech_clean["fishingtech_category_name"]=fishingtech_clean["fishingtech_code"].map(fishingtech_name_map).fillna("INACTIVE")

# Visualisation des nouvelles variables
fishingtech_cols=["fishingtech_code","fishingtech_name","fishingtech_category_code","fishingtech_category_name","origin_sheet"]
fishingtech_clean=fishingtech_clean.reindex(columns=fishingtech_cols).sort_values(by="fishingtech_code")
display(fishingtech_clean)

,fishingtech_code,fishingtech_name,fishingtech_category_code,fishingtech_category_name,origin_sheet
0,DFN,Drift and/or fixed netters,PG,"Passive gears (nets, lines, traps)",additional info
1,DRB,Dredgers,Dre,Dredgers and polyvalent,additional info
2,DTS,Demersal trawlers and/or demersal seiners,TraD,Demersal trawlers and seiners,additional info
3,FPO,Vessels using pots and/or traps,PG,"Passive gears (nets, lines, traps)",additional info
4,HOK,Vessels using hooks,PG,"Passive gears (nets, lines, traps)",additional info
6,MGO,Vessel using other active gears,PG,"Passive gears (nets, lines, traps)",additional info
7,MGP,Vessels using polyvalent active gears only,Dre,Dredgers and polyvalent,additional info
8,PG,Passive Gears,PG,"Passive gears (nets, lines, traps)",additional info
9,PGO,Vessels using other passive gears,PG,"Passive gears (nets, lines, traps)",additional info
10,PGP,Vessels using polyvalent passive gears only,PG,"Passive gears (nets, lines, traps)",additional info


#### 4.1.3 Création de la table de dimension vessel_clean

Les navires vont être regroupé en 4 catégories selon les référentiels de la pêche :
* **Inshore :** Côtier
* **Offshore :** Hauturier
* **Industrial :** Insdustriel
* **Other :** Autres

In [116]:
# Assignation de la variable vessel_clean
vessel_clean=landings_clean["vessel_length"].drop_duplicates().sort_values().reset_index(drop=True).to_frame()

# Dictionnaire de référence
dict_vessel={
    "Inshore":("VL0006","VL0008","VL0010","VL0612","VL0812","VL1012"),
    "Offshore":("VL1218","VL1824"),
    "Industrial":("VL2440","VL40XX")
}

# Dictionnaire de correspondance
vessel_code_map={code: key for key,codes in dict_vessel.items() for code in codes}

# Mapping sur le DataFrame
vessel_clean["vessel_category"]=vessel_clean["vessel_length"].map(vessel_code_map)

# Visualisation des nouvelles variables
display(vessel_clean)


,vessel_length,vessel_category
0,VL0006,Inshore
1,VL0008,Inshore
2,VL0010,Inshore
3,VL0612,Inshore
4,VL0812,Inshore
5,VL1012,Inshore
6,VL1218,Offshore
7,VL1824,Offshore
8,VL2440,Industrial
9,VL40XX,Industrial


#### 4.1.4 Création de la table de dimension geozone_clean

Les deux tables de faits ne présentent pas la même granularité géographique. Bien qu’elles partagent le niveau `supra_reg`, la table `landings_clean` dispose d’un niveau de détail supplémentaire avec `sub_reg`. Afin d’harmoniser le modèle et de garantir des jointures cohérentes entre les faits, une table de dimension `geozone_clean` est créée autour du niveau géographique commun supra_reg.

In [117]:
display(fs_clean["supra_reg"].unique())
display(landings_clean["supra_reg"].unique())

array(['AREA27', 'MBS', 'AREA37', 'NAO', 'OFR'], dtype=object)

array(['AREA27', 'MBS', 'AREA37', 'OFR', 'NAO'], dtype=object)

In [118]:
# A GARDER DANS LE CAS OU UNE NOUVELLE ZONE APPARAIT
#"AREA18":"Arctic Sea",
#"AREA21":"Atlantic, Northwest",
#"AREA31":"Atlantic, Western Central",
#"AREA34":"Atlantic, Eastern Central",
#"AREA41":"Atlantic, Southwest",
#"AREA47":"Atlantic, Southeast",
#"AREA48":"Atlantic, Antarctic",
#"AREA51":"Indian Ocean, Western",
#"AREA57":"Indian Ocean, Eastern",
#"AREA58":"Indian Ocean, Antarctic and Southern",
#"AREA61":"Pacific, Northwest",
#"AREA67":"Pacific, Northeast",
#"AREA71":"Pacific, Western Central",
#"AREA77":"Pacific, Eastern Central",
#"AREA81":"Pacific, Southwest",
#"AREA87":"Pacific, Southeast",
#"AREA88":"Pacific, Antarctic",

# Dictionnaire de correspondance
dict_geozone={
    "AREA27":"Atlantic, Northeast",
    "AREA37":"Mediterranean and Black Sea",
    "OFR":"Other fishing regions"
}

# Visualisation technique des nouvelles variables
geozone_clean=pd.Series(dict_geozone).to_frame(name="area_name").reset_index().rename(columns={"index":"area_code"})
check_data_quality(geozone_clean)
geozone_clean.head()

--------------------------------------------------------------- Visualisation technique de Dataset ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,area_code,object,0,0,0.00%,3,"['AREA27', 'AREA37', 'OFR']"
1,area_name,object,0,0,0.00%,3,"['Atlantic, Northeast', 'Mediterranean and Black Sea', 'Other fishing regions']"
2,--- GLOBAL ---,-,0,0,0.00%,-,3 lignes et 2 colonnes


,area_code,area_name
0,AREA27,"Atlantic, Northeast"
1,AREA37,Mediterranean and Black Sea
2,OFR,Other fishing regions


#### 4.1.5 Création de la table de dimension variable_clean

Les nomenclatures issues de fs_clean et landings_clean sont centralisés dans un référentiel unique (variable_clean). En normalisant, dédoublonnant et triant les données, cette approche garantit une structure légère et optimisée, indispensable pour maximiser la vitesse de traitement et la fluidité des relations au sein de votre modèle Power BI

In [119]:
# Création de la table variable_clean
fs_variable=fs_clean[["variable_code","variable_name","variable_group"]].drop_duplicates().reset_index(drop=True)
landings_variable=landings_clean[["variable_code","variable_name","variable_group"]].drop_duplicates().reset_index(drop=True)
variable_clean=pd.concat([fs_variable,landings_variable],axis=0).drop_duplicates().sort_values(by="variable_code").reset_index(drop=True)

# Visualisation technique des nouvelles variables
check_data_quality(variable_clean)
variable_clean.head()

--------------------------------------------------------------- Visualisation technique de Dataset ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,variable_code,object,0,0,0.00%,38,"['assets', 'avgage', 'avgloa']"
1,variable_name,object,0,0,0.00%,38,"['Total assets', 'Mean age of vessels', 'Mean LOA of vessels']"
2,variable_group,object,31,0,0.00%,7,"['Capital', 'Capacity', 'Effort']"
3,--- GLOBAL ---,-,0,0,0.00%,-,38 lignes et 3 colonnes


,variable_code,variable_name,variable_group
0,assets,Total assets,Capital
1,avgage,Mean age of vessels,Capacity
2,avgloa,Mean LOA of vessels,Capacity
3,debts,Gross debt,Capital
4,gtseadays,GT days at sea,Effort


### 4.2 Harmonisation des référentiels (mapping) entre les deux tables de faits et les tables de dimensions

Cette étape assure la cohérence entre les colonnes de liaison (clés étrangères) des tables de faits et les référentiels des tables de dimensions (clés primaires). L'objectif est de garantir une intégrité référentielle totale pour éviter les erreurs de jointure dans Power BI.

In [120]:
# Définition de la fonction pour vérifier les référentiels
def check_referential_integrity(fact_df,fact_col,dim_df,dim_col):
    # Récupération des valeurs uniques
    fact_set=set(fact_df[fact_col].unique())
    dim_set=set(dim_df[dim_col].unique())
    
    # Calcul des codes orphelins (présents dans faits mais pas dans dim)
    orphans=fact_set-dim_set
    
    print(f"--- Vérification sur la clé : {fact_col} ---")
    if not orphans:
        print("✅ Succès : Tous les codes de la table de faits existent dans la dimension.")
        print(f"Nombre de codes uniques vérifiés : {len(fact_set)}")
    else:
        print(f"❌ Alerte : {len(orphans)} code(s) n'ont pas de correspondance dans la dimension !")
        print(f"Codes orphelins : {orphans}")
        
    # Optionnel : Calcul des codes inutilisés (présents dans dim mais pas dans faits)
    unused=dim_set-fact_set
    if unused:
        print(f"ℹ️ Info : {len(unused)} code(s) de la dimension ne sont pas utilisés dans les faits.")

    print("-" * 100)

#### 4.2.1 Harmonisation des référentiels (mapping) entre landings_clean et les tables de dimensions

Le processus consiste à auditer systématiquement les clés de liaison entre la table de faits et les dimensions pour corriger les éventuelles ruptures d'intégrité. L'alignement des codes (ex: NAO/MBS vers AREA27/37) assure une correspondance de 100 % entre les tables, condition indispensable à la stabilité du modèle relationnel sous Power BI.

In [121]:
# Vérification des référentiels
check_referential_integrity(landings_clean,"country_code",country_clean,"country_code")
check_referential_integrity(landings_clean,"supra_reg",geozone_clean,"area_code")
check_referential_integrity(landings_clean,"fishing_tech",fishingtech_clean,"fishingtech_code")
check_referential_integrity(landings_clean,"vessel_length",vessel_clean,"vessel_length")
check_referential_integrity(landings_clean,"species_code",species_clean,"species_code")
check_referential_integrity(landings_clean,"variable_code",variable_clean,"variable_code")

# Remplacement du nom du code par le code d'origine
landings_clean["supra_reg"]=landings_clean["supra_reg"].replace({"NAO":"AREA27","MBS":"AREA37"})

# Vérification de la correction
check_referential_integrity(landings_clean,"supra_reg",geozone_clean,"area_code")

--- Vérification sur la clé : country_code ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 22
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : supra_reg ---
❌ Alerte : 2 code(s) n'ont pas de correspondance dans la dimension !
Codes orphelins : {'NAO', 'MBS'}
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : fishing_tech ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 14
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : vessel_length ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 10
----------------------------------------------------------------------

Lors du contrôle qualité des données de landings_clean, 12 lignes en double persistent, dont la somme de la variable `value` est égale à 0. Comme souvent observé, ces doublons proviennent de l’Espagne. Étant donné que ces 12 lignes ont une somme nulle, elles seront supprimées.

In [122]:
# Visualisation des doublons de landings_clean
display(landings_clean[landings_clean.duplicated(keep=False).sort_values()])

# Suppression des 12 lignes en doublon
landings_clean=landings_clean.drop_duplicates()

# Visualisation technique de landings_clean
check_data_quality(landings_clean,"landings_clean")

/var/folders/y4/sbgjvv4n3xz80b3yl21db8l80000gn/T/ipykernel_2834/2655993145.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  display(landings_clean[landings_clean.duplicated(keep=False).sort_values()])


,origin_sheet,fs_name,year,country_code,country_name,supra_reg,sub_reg,fishing_tech,vessel_length,variable_group,variable_code,variable_name,species_code,species_name,unit,value
1171359,Landings 2017-,ESP MBS HOK1218 LLD *,2018,ESP,Spain,AREA37,sa 5,HOK,VL1218,Landings,totvallandg,Value of landings,OTH,Other,euro,0.0
1171360,Landings 2017-,ESP MBS HOK1218 LLD *,2018,ESP,Spain,AREA37,sa 5,HOK,VL1218,Landings,totvallandg,Value of landings,OTH,Other,euro,0.0
1171551,Landings 2017-,ESP MBS HOK1218 LLD *,2018,ESP,Spain,AREA37,sa 6,HOK,VL1218,Landings,totvallandg,Value of landings,BRA,Brama spp,euro,0.0
1171552,Landings 2017-,ESP MBS HOK1218 LLD *,2018,ESP,Spain,AREA37,sa 6,HOK,VL1218,Landings,totvallandg,Value of landings,BRA,Brama spp,euro,0.0
1171630,Landings 2017-,ESP MBS HOK1218 LLD *,2018,ESP,Spain,AREA37,sa 6,HOK,VL1218,Landings,totvallandg,Value of landings,OTH,Other,euro,0.0
1171631,Landings 2017-,ESP MBS HOK1218 LLD *,2018,ESP,Spain,AREA37,sa 6,HOK,VL1218,Landings,totvallandg,Value of landings,OTH,Other,euro,0.0
1172581,Landings 2017-,ESP MBS HOK1218 LLD *,2019,ESP,Spain,AREA37,sa 5,HOK,VL1218,Landings,totvallandg,Value of landings,BRA,Brama spp,euro,0.0
1172582,Landings 2017-,ESP MBS HOK1218 LLD *,2019,ESP,Spain,AREA37,sa 5,HOK,VL1218,Landings,totvallandg,Value of landings,BRA,Brama spp,euro,0.0
1172604,Landings 2017-,ESP MBS HOK1218 LLD *,2019,ESP,Spain,AREA37,sa 5,HOK,VL1218,Landings,totvallandg,Value of landings,OTH,Other,euro,0.0
1172605,Landings 2017-,ESP MBS HOK1218 LLD *,2019,ESP,Spain,AREA37,sa 5,HOK,VL1218,Landings,totvallandg,Value of landings,OTH,Other,euro,0.0


--------------------------------------------------------------- Visualisation technique de landings_clean ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,origin_sheet,object,1 671 227,0,0.00%,3,"['Landings 2008-2012', 'Landings 2013-2017', 'Landings 2017-']"
1,fs_name,object,1 670 587,0,0.00%,643,"['BEL NAO DTS2440 NGI *', 'BEL NAO PMP1824 NGI *', 'BEL NAO TBB1824 NGI *']"
2,year,int64,1 671 215,0,0.00%,15,"[2008, 2009, 2010]"
3,country_code,object,1 671 208,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
4,country_name,object,1 671 208,0,0.00%,22,"['Belgium', 'Bulgaria', 'Cyprus']"
5,supra_reg,object,1 671 227,0,0.00%,3,"['AREA27', 'AREA37', 'OFR']"
6,sub_reg,object,1 670 977,0,0.00%,253,"['27.4.b', '27.4.c', '27.7.a']"
7,fishing_tech,object,1 671 216,0,0.00%,14,"['DTS', 'PMP', 'TBB']"
8,vessel_length,object,1 671 220,0,0.00%,10,"['VL2440', 'VL1824', 'VL0006']"
9,variable_group,object,1 671 229,0,0.00%,1,['Landings']


L'audit confirme que la table de faits `landings_clean` est alignée avec l'ensemble des tables de dimensions.

#### 4.2.2 Harmonisation des référentiels (mapping) entre fs_clean et les tables de dimensions

L'audit des clés de liaison permet de détecter les éventuels codes orphelins entre la table de faits et les dimensions. Par une phase d'harmonisation (ex: conversion de NAO/MBS en AREA27/37), les référentiels sont alignés pour garantir une correspondance de 100 %, assurant ainsi la fiabilité des jointures dans le modèle final.
Concernant la table `energy_excise_duty_clean`, les écarts résiduels (codes orphelins) identifiés sur les périodes antérieures à 2013 ne sont pas des erreurs techniques mais reflètent les limites temporelles des jeux de données sources. Cette distinction permet d'assurer une parfaite transparence sur la portée analytique du modèle final.

In [123]:
# Ajout des clés primaires/étrangères pour assurer l'intégrité relationnelle entre energy_excise_duty_clean et fs_clean
energy_excise_duty_clean.insert(0,"country_year",energy_excise_duty_clean["country_code"].astype(str)+"_"+energy_excise_duty_clean["year"].astype(str))
energy_excise_duty_clean=energy_excise_duty_clean.sort_values(by="country_year")
check_data_quality(energy_excise_duty_clean)

fs_clean.insert(2,"country_year",fs_clean["country_code"].astype(str)+"_"+fs_clean["year"].astype(str))
check_data_quality(energy_excise_duty_clean)

# Vérification des référentiels
check_referential_integrity(fs_clean,"country_code",country_clean,"country_code")
check_referential_integrity(fs_clean,"fishing_tech",fishingtech_clean,"fishingtech_code")
check_referential_integrity(fs_clean,"vessel_length",vessel_clean,"vessel_length")
check_referential_integrity(fs_clean,"supra_reg",geozone_clean,"area_code")
check_referential_integrity(fs_clean,"variable_code",variable_clean,"variable_code")
check_referential_integrity(fs_clean,"country_year",energy_excise_duty_clean,"country_year")

# Remplacement du nom du code par le code d'origine
fs_clean["supra_reg"]=fs_clean["supra_reg"].replace({"NAO":"AREA27","MBS":"AREA37"})

# Vérification de la correction
check_referential_integrity(fs_clean,"supra_reg",geozone_clean,"area_code")


--------------------------------------------------------------- Visualisation technique de Dataset ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,country_year,object,0,0,0.00%,278,"['AUT_2013', 'AUT_2014', 'AUT_2015']"
1,year,int64,268,0,0.00%,10,"[2013, 2014, 2015]"
2,country_code,object,250,0,0.00%,28,"['AUT', 'BEL', 'BGR']"
3,country_name,object,250,0,0.00%,28,"['Austria', 'Belgium', 'Bulgaria']"
4,excise_duty_eur_l,float64,128,0,0.00%,150,"[0.4277, 0.4096, 0.4051]"
5,origin_sheet,object,277,0,0.00%,1,['Feuil1']
6,--- GLOBAL ---,-,0,0,0.00%,-,278 lignes et 6 colonnes


--------------------------------------------------------------- Visualisation technique de Dataset ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,country_year,object,0,0,0.00%,278,"['AUT_2013', 'AUT_2014', 'AUT_2015']"
1,year,int64,268,0,0.00%,10,"[2013, 2014, 2015]"
2,country_code,object,250,0,0.00%,28,"['AUT', 'BEL', 'BGR']"
3,country_name,object,250,0,0.00%,28,"['Austria', 'Belgium', 'Bulgaria']"
4,excise_duty_eur_l,float64,128,0,0.00%,150,"[0.4277, 0.4096, 0.4051]"
5,origin_sheet,object,277,0,0.00%,1,['Feuil1']
6,--- GLOBAL ---,-,0,0,0.00%,-,278 lignes et 6 colonnes


--- Vérification sur la clé : country_code ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 22
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : fishing_tech ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 14
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : vessel_length ---
✅ Succès : Tous les codes de la table de faits existent dans la dimension.
Nombre de codes uniques vérifiés : 10
----------------------------------------------------------------------------------------------------
--- Vérification sur la clé : supra_reg ---
❌ Alerte : 2 code(s) n'ont pas de correspondance dans la dimension !
Codes orphelins : {'NAO', 'MBS'}
----------------------------------------------------------------------

Afin de conclure l’évaluation de la qualité des données, une vérification technique est réalisée pour en valider la fiabilité.

In [124]:
# Visualisation technique de fs_clean
check_data_quality(fs_clean)

--------------------------------------------------------------- Visualisation technique de Dataset ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,origin_sheet,object,295 833,0,0.00%,1,['FS data']
1,fs_name,object,295 170,0,0.00%,664,"['BEL NAO DTS2440 NGI *', 'BEL NAO TBB1824 NGI *', 'BGR MBS TM 1824 NGI']"
2,country_year,object,295 508,0,0.00%,326,"['BEL_2008', 'BEL_2011', 'BEL_2010']"
3,year,int64,295 819,0,0.00%,15,"[2008, 2011, 2010]"
4,country_code,object,295 812,0,0.00%,22,"['BEL', 'BGR', 'GRC']"
5,country_name,object,295 812,0,0.00%,22,"['Belgium', 'Bulgaria', 'Greece']"
6,supra_reg,object,295 831,0,0.00%,3,"['AREA27', 'AREA37', 'OFR']"
7,fishing_tech,object,295 820,0,0.00%,14,"['DTS', 'TBB', 'TM']"
8,vessel_length,object,295 824,0,0.00%,10,"['VL2440', 'VL1012', 'VL1824']"
9,variable_group,object,295 828,0,0.00%,6,"['Expenditure', 'Effort', 'Capital']"


L'audit confirme que la table de faits `fs_clean` est alignée avec l'ensemble des tables de dimensions.

### 4.3 Exportation du modèle relationnel

Cette étape finale consiste à exporter les tables de faits et de dimensions au format CSV. Le jeu de données est optimisé en supprimant les colonnes redondantes et non pertinentes afin d'alléger le modèle de données pour l'analyse sous Power BI

In [125]:
# Exportation des tables de faits
fact_fs=fs_clean.drop(columns=["country_name","variable_name","variable_group"])
fact_fs.to_csv("Data/Processed/fact_fs.csv",index=False,encoding="utf-8-sig",sep=",")

fact_landings=landings_clean.drop(columns=["country_name","variable_name","variable_group","species_name"])
fact_landings.to_csv("Data/Processed/fact_landings.csv",index=False,encoding="utf-8-sig",sep=",")

# Exportation des tables de dimensions
dim_species=species_clean
dim_species.to_csv("Data/Processed/dim_species.csv",index=False,encoding="utf-8-sig",sep=",")

dim_fishingtech=fishingtech_clean
dim_fishingtech.to_csv("Data/Processed/dim_fishingtech.csv",index=False,encoding="utf-8-sig",sep=",")

dim_vessel=vessel_clean
dim_vessel.to_csv("Data/Processed/dim_vessel.csv",index=False,encoding="utf-8-sig",sep=",")

dim_geozone=geozone_clean
dim_geozone.to_csv("Data/Processed/dim_geozone.csv",index=False,encoding="utf-8-sig",sep=",")

dim_country=country_clean
dim_country.to_csv("Data/Processed/dim_country.csv",index=False,encoding="utf-8-sig",sep=",")

dim_variable=variable_clean
dim_variable.to_csv("Data/Processed/dim_variable.csv",index=False,encoding="utf-8-sig",sep=",")

dim_energy_excise_duty=energy_excise_duty_clean
dim_energy_excise_duty.to_csv("Data/Processed/dim_energy_excise_duty.csv",index=False,encoding="utf-8-sig",sep=",")


In [126]:
# Contrôle de qualité des tables de faits après export en cvs
check_data_quality(fact_fs,"fact_fs")
check_data_quality(fact_landings,"fact_landings")

# Contrôle de qualité des tables de dimensions après export en cvs
check_data_quality(dim_country,"dim_country")
check_data_quality(dim_fishingtech,"dim_fishingtech")
check_data_quality(dim_geozone,"dim_geozone")
check_data_quality(dim_species,"dim_species")
check_data_quality(dim_vessel,"dim_vessel")
check_data_quality(dim_vessel,"dim_variable")
check_data_quality(dim_energy_excise_duty,"dim_energy_excise_duty")

--------------------------------------------------------------- Visualisation technique de fact_fs ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,origin_sheet,object,295 833,0,0.00%,1,['FS data']
1,fs_name,object,295 170,0,0.00%,664,"['BEL NAO DTS2440 NGI *', 'BEL NAO TBB1824 NGI *', 'BGR MBS TM 1824 NGI']"
2,country_year,object,295 508,0,0.00%,326,"['BEL_2008', 'BEL_2011', 'BEL_2010']"
3,year,int64,295 819,0,0.00%,15,"[2008, 2011, 2010]"
4,country_code,object,295 812,0,0.00%,22,"['BEL', 'BGR', 'GRC']"
5,supra_reg,object,295 831,0,0.00%,3,"['AREA27', 'AREA37', 'OFR']"
6,fishing_tech,object,295 820,0,0.00%,14,"['DTS', 'TBB', 'TM']"
7,vessel_length,object,295 824,0,0.00%,10,"['VL2440', 'VL1012', 'VL1824']"
8,variable_code,object,295 798,0,0.00%,36,"['totdepcost', 'totseadays', 'totinvest']"
9,unit,object,295 823,0,0.00%,11,"['euro', 'day', 'metre']"


--------------------------------------------------------------- Visualisation technique de fact_landings ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,origin_sheet,object,1 671 227,0,0.00%,3,"['Landings 2008-2012', 'Landings 2013-2017', 'Landings 2017-']"
1,fs_name,object,1 670 587,0,0.00%,643,"['BEL NAO DTS2440 NGI *', 'BEL NAO PMP1824 NGI *', 'BEL NAO TBB1824 NGI *']"
2,year,int64,1 671 215,0,0.00%,15,"[2008, 2009, 2010]"
3,country_code,object,1 671 208,0,0.00%,22,"['BEL', 'BGR', 'CYP']"
4,supra_reg,object,1 671 227,0,0.00%,3,"['AREA27', 'AREA37', 'OFR']"
5,sub_reg,object,1 670 977,0,0.00%,253,"['27.4.b', '27.4.c', '27.7.a']"
6,fishing_tech,object,1 671 216,0,0.00%,14,"['DTS', 'PMP', 'TBB']"
7,vessel_length,object,1 671 220,0,0.00%,10,"['VL2440', 'VL1824', 'VL0006']"
8,variable_code,object,1 671 228,0,0.00%,2,"['totvallandg', 'totwghtlandg']"
9,species_code,object,1 665 747,0,0.00%,5 483,"['ANF', 'BIB', 'BLL']"


--------------------------------------------------------------- Visualisation technique de dim_country ---------------------------------------------------------------


...

,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,vessel_length,object,0,0,0.00%,10,"['VL0006', 'VL0008', 'VL0010']"
1,vessel_category,object,7,0,0.00%,3,"['Inshore', 'Offshore', 'Industrial']"
2,--- GLOBAL ---,-,0,0,0.00%,-,10 lignes et 2 colonnes


--------------------------------------------------------------- Visualisation technique de dim_variable ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,vessel_length,object,0,0,0.00%,10,"['VL0006', 'VL0008', 'VL0010']"
1,vessel_category,object,7,0,0.00%,3,"['Inshore', 'Offshore', 'Industrial']"
2,--- GLOBAL ---,-,0,0,0.00%,-,10 lignes et 2 colonnes


--------------------------------------------------------------- Visualisation technique de dim_energy_excise_duty ---------------------------------------------------------------


,VARIABLE,TYPE,DOUBLON,NA,% NA,MODALITE,APERCU
0,country_year,object,0,0,0.00%,278,"['AUT_2013', 'AUT_2014', 'AUT_2015']"
1,year,int64,268,0,0.00%,10,"[2013, 2014, 2015]"
2,country_code,object,250,0,0.00%,28,"['AUT', 'BEL', 'BGR']"
3,country_name,object,250,0,0.00%,28,"['Austria', 'Belgium', 'Bulgaria']"
4,excise_duty_eur_l,float64,128,0,0.00%,150,"[0.4277, 0.4096, 0.4051]"
5,origin_sheet,object,277,0,0.00%,1,['Feuil1']
6,--- GLOBAL ---,-,0,0,0.00%,-,278 lignes et 6 colonnes


Aucun doublon ni valeur manquante n’est présent dans les données, et chaque table de dimension dispose bien d’une clé primaire unique.
Le nettoyage des données est désormais terminé, et l’étape de modélisation sous Power BI peut commencer.